In [ ]:
# ============================================================
# CELL 1: Environment Setup, Installations, and Central Imports
# ============================================================

# 1. Mount Google Drive to access project files and datasets
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Install external dependencies (Quiet mode)
print("--- 🛠️  Installing dependencies... ---")
# Audio compression, processing, and fast data loading libraries
!pip install -q encodec torchaudio tqdm torchcodec
# State-of-the-art audio generation framework from Meta
!pip install -q git+https://github.com/facebookresearch/audiocraft
# Tool for handling various audio/video container formats
!pip install -q av
print("✅ Installation completed!")

# 3. Standard Library Imports
import os           # Operating system interfaces (file/folder management)
import glob         # Unix style pathname pattern expansion (finding files)
import json         # JSON encoder and decoder (for text captions)
import shutil       # High-level file operations (copying/moving)
import re           # Regular expression operations (parsing filenames)
import warnings     # Control over warning messages

# 4. Data Science & Visualization
import numpy as np              # Fundamental package for numerical computing
import matplotlib.pyplot as plt # Plotting and visualization
import librosa                  # Feature extraction and music/audio analysis
import librosa.display          # Specialized display tools for audio
from tqdm.auto import tqdm       # Smart progress bars for loops
from IPython.display import Audio, display # Interactive audio player in Colab

# 5. Deep Learning & Audio Processing
import torch                    # Core PyTorch library for neural networks
import torchaudio               # PyTorch library for audio signal processing
from transformers import T5Tokenizer, T5EncoderModel # NLP models for text conditioning
from encodec import EncodecModel # Neural audio codec for latent representation
from encodec.utils import convert_audio # Utility to match model's audio specs

# --- Global Configurations ---
warnings.filterwarnings('ignore') # Suppress non-critical warnings for cleaner output
# Set device to GPU if available, else fallback to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n🚀 System ready. Using device: {device.upper()}")

Mounted at /content/drive
--- 🛠️  Installing dependencies... ---
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 38.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 37.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 68.4 MB/s eta 0:00:00
✅ Installation completed!

🚀 System ready. Using device: 

In [ ]:
# Define the base path for the project in Google Drive.
BASE_DIR = '/content/drive/MyDrive/MasterProject'

# List of required directories for data, features, and outputs.
folders = [
    "data/raw/mimii_dg",          # For downloaded raw data (ZIPs)
    "data/splits/train",          # For sorted train audio files
    "data/splits/val",            # For sorted validation audio files
    "data/splits/test",           # For sorted test audio files
    "data/features/encodec/train",# For processed "latents" (training)
    "data/features/encodec/val",  # For processed "latents" (validation)
    "outputs/logs",               # For training logs (e.g., TensorBoard)
    "outputs/checkpoints",        # To save trained models (.pth)
    "outputs/samples",            # To save generated audio samples
    "configs"                     # For configuration files (e.g., hyperparameters)
]

Folder structure created/verified in /content/drive/MyDrive/MasterProject


In [ ]:
# Split DEVELOPMENT Data (900/90/10 Logic)


# Configuration
BASE_DIR = os.environ.get('BASE_DIR', '/content/drive/MyDrive/MasterProject')
if not os.path.exists(BASE_DIR):
    raise ValueError(f'ERROR: BASE_DIR not found at {BASE_DIR}.')

# Source Base Folder
DEV_DATA_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'mimii_dg')

# Target paths
TARGET_TRAIN_DIR_BASE = os.path.join(BASE_DIR, 'data', 'splits', 'train')
TARGET_VAL_DIR_BASE = os.path.join(BASE_DIR, 'data', 'splits', 'val')

# The "clean" machine names and their folder names
MACHINE_TYPES_DEV = ['dev_bearing', 'dev_fan', 'dev_gearbox', 'dev_slider', 'dev_ToyCar', 'dev_ToyTrain', 'dev_valve']
MACHINE_NAMES_CLEAN = [m.replace('dev_', '') for m in MACHINE_TYPES_DEV]

# Split configuration (per section)
# Every section has 1000 files
# divided in 990 source domain files and 10 target domain files
# 900 of the 990 source domain files goes into the train split folder
N_TRAIN_SOURCE = 900
# 90 of the 990 source domain goes into the val split folder
N_VAL_SOURCE = 90
# Reproducibility:
# Ensure random shuffling is always the same
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Counter for copied files
total_files_copied_train = 0
total_files_copied_val = 0

# main-loop for each machine
for machine_folder, machine_clean in zip(MACHINE_TYPES_DEV, MACHINE_NAMES_CLEAN):

    # Define the paths
    # Source path: .../mimii_dg/dev_bearing/bearing/train/
    SOURCE_DIR = os.path.join(DEV_DATA_DIR, machine_folder, machine_clean, 'train')

    # Target path:
    target_train_dir = os.path.join(TARGET_TRAIN_DIR_BASE, machine_clean)
    target_val_dir = os.path.join(TARGET_VAL_DIR_BASE, machine_clean)

    # Create folders if exist_ok=False
    os.makedirs(target_train_dir, exist_ok=True)
    os.makedirs(target_val_dir, exist_ok=True)

    # Finding the files
    print(f"Searching for .wav files in: {SOURCE_DIR}")
    all_wav_files = glob.glob(os.path.join(SOURCE_DIR, "*.wav"))

    if not all_wav_files:
        print(f"No development files found for {machine_clean}. Check path or sync.")
        continue

    # Loop through sections 00 01 and 02
    sections = ["section_00", "section_01", "section_02"]

    for section in sections:
        # Filter files for these sections
        section_files = [f for f in all_wav_files if section in os.path.basename(f)]

        if not section_files:
            print(f"No files found for {section}")
            continue

        # Divide source and target files (into a list)
        source_files = [f for f in section_files if "_source_" in os.path.basename(f)]
        target_files = [f for f in section_files if "_target_" in os.path.basename(f)]

        print(f"  {section}: Found {len(source_files)} Source, {len(target_files)} Target.")

        # 900/90/10 seperation
        np.random.shuffle(source_files)

        # A) 900 Source -> Train
        split_train_source = source_files[:N_TRAIN_SOURCE]
        # B) 90 Source -> Val
        split_val_source = source_files[N_TRAIN_SOURCE : N_TRAIN_SOURCE + N_VAL_SOURCE]
        # C) 10 Target -> Train
        split_train_target = target_files

        # Merge lists
        final_train_list = split_train_source + split_train_target
        final_val_list = split_val_source

        # Copy files of train
        for f_path in final_train_list:
            filename = os.path.basename(f_path)
            target_path = os.path.join(target_train_dir, filename)
            if not os.path.exists(target_path):
                shutil.copy(f_path, target_path)
                total_files_copied_train += 1

        # Copy files of validation
        for f_path in final_val_list:
            filename = os.path.basename(f_path)
            target_path = os.path.join(target_val_dir, filename)
            if not os.path.exists(target_path):
                shutil.copy(f_path, target_path)
                total_files_copied_val += 1

    print(f"Processed {machine_clean} (Dev).")

print(f"\n\n DEVELOPMENT SPLIT COMPLETE.")
print(f"Total copied to 'train': {total_files_copied_train}")
print(f"Total copied to 'val': {total_files_copied_val}")

Searching for .wav files in: /content/drive/MyDrive/MasterProject/data/raw/mimii_dg/dev_bearing/bearing/train
  section_00: Found 990 Source, 10 Target.
  section_01: Found 990 Source, 9 Target.
  section_02: Found 990 Source, 10 Target.
Processed bearing (Dev).
Searching for .wav files in: /content/drive/MyDrive/MasterProject/data/raw/mimii_dg/dev_fan/fan/train
  section_00: Found 990 Source, 10 Target.
  section_01: Found 990 Source, 10 Target.
  section_02: Found 990 Source, 10 Target.
Processed fan (Dev).
Searching for .wav files in: /content/drive/MyDrive/MasterProject/data/raw/mimii_dg/dev_gearbox/gearbox/train
  section_00: Found 990 Source, 10 Target.
  section_01: Found 990 Source, 10 Target.
  section_02: Found 990 Source, 10 Target.
Processed gearbox (Dev).
Searching for .wav files in: /content/drive/MyDrive/MasterProject/data/raw/mimii_dg/dev_slider/slider/train
  section_00: Found 990 Source, 10 Target.
  section_01: Found 990 Source, 10 Target.
  section_02: Found 990 Sou

KeyboardInterrupt: 

In [ ]:
# Split ADDITIONAL Data (900/90/10 Logic)

# Configuration
BASE_DIR = os.environ.get('BASE_DIR', '/content/drive/MyDrive/MasterProject')
if not os.path.exists(BASE_DIR):
    raise ValueError(f'ERROR: BASE_DIR not found at {BASE_DIR}.')

# Source Base Folder
ADDITIONAL_DATA_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'mimii_dg', 'additional_training_dataset')

# Target paths
TARGET_TRAIN_ADDITIONAL_BASE = os.path.join(BASE_DIR, 'data', 'splits', 'train_additional')
TARGET_VAL_ADDITIONAL_BASE = os.path.join(BASE_DIR, 'data', 'splits', 'val_additional')

# The "clean" machine names
MACHINE_NAMES_CLEAN = ['bearing', 'fan', 'gearbox', 'slider', 'ToyCar', 'ToyTrain', 'valve']
# The specific folder names for additional data (usually 'eval_data_bearing_train', etc.)
MACHINE_FOLDERS_ADD = [f'eval_data_{m}_train' for m in MACHINE_NAMES_CLEAN]

# Split configuration (per section)
# 900 of the source domain files go into the train split folder
N_TRAIN_SOURCE = 900
# 90 of the source domain files go into the val split folder
N_VAL_SOURCE = 90
# Reproducibility:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Counter for copied files
total_files_copied_train = 0
total_files_copied_val = 0

# main-loop for each machine
for machine_folder, machine_clean in zip(MACHINE_FOLDERS_ADD, MACHINE_NAMES_CLEAN):

    # Define the paths
    # Source path: .../additional_training_dataset/eval_data_bearing_train/bearing/train/
    SOURCE_DIR = os.path.join(ADDITIONAL_DATA_DIR, machine_folder, machine_clean, 'train')

    # Target path:
    target_train_dir = os.path.join(TARGET_TRAIN_ADDITIONAL_BASE, machine_clean)
    target_val_dir = os.path.join(TARGET_VAL_ADDITIONAL_BASE, machine_clean)

    # Create folders if exist_ok=False
    os.makedirs(target_train_dir, exist_ok=True)
    os.makedirs(target_val_dir, exist_ok=True)

    # Finding the files
    print(f"Searching for .wav files in: {SOURCE_DIR}")
    all_wav_files = glob.glob(os.path.join(SOURCE_DIR, "*.wav"))

    if not all_wav_files:
        print(f"No additional files found for {machine_clean}. Check path or sync.")
        continue

    # Loop through sections 03, 04 and 05 (Specific to Additional Dataset)
    sections = ["section_03", "section_04", "section_05"]

    for section in sections:
        # Filter files for these sections
        section_files = [f for f in all_wav_files if section in os.path.basename(f)]

        if not section_files:
            print(f"No files found for {section} in {machine_clean}")
            continue

        # Divide source and target files (into a list)
        # We also check for '_normal_' to be safe
        source_files = [f for f in section_files if "_source_" in os.path.basename(f) and "_normal_" in os.path.basename(f)]
        target_files = [f for f in section_files if "_target_" in os.path.basename(f) and "_normal_" in os.path.basename(f)]

        print(f"  {section}: Found {len(source_files)} Source, {len(target_files)} Target.")

        # Safety check: Ensure enough files exist
        if len(source_files) < (N_TRAIN_SOURCE + N_VAL_SOURCE):
            print(f"    WARNING: Not enough source files in {section}! Taking all available.")
            current_n_train = int(len(source_files) * 0.9)

            np.random.shuffle(source_files)

            split_train_source = source_files[:current_n_train]
            split_val_source = source_files[current_n_train:]
            split_train_target = target_files
        else:
            # 900/90/10 separation
            np.random.shuffle(source_files)

            # A) 900 Source -> Train
            split_train_source = source_files[:N_TRAIN_SOURCE]
            # B) 90 Source -> Val
            split_val_source = source_files[N_TRAIN_SOURCE : N_TRAIN_SOURCE + N_VAL_SOURCE]
            # C) All Target -> Train
            split_train_target = target_files

        # Merge lists
        final_train_list = split_train_source + split_train_target
        final_val_list = split_val_source

        # Copy files of train
        for f_path in final_train_list:
            filename = os.path.basename(f_path)
            target_path = os.path.join(target_train_dir, filename)
            if not os.path.exists(target_path):
                shutil.copy(f_path, target_path)
                total_files_copied_train += 1

        # Copy files of validation
        for f_path in final_val_list:
            filename = os.path.basename(f_path)
            target_path = os.path.join(target_val_dir, filename)
            if not os.path.exists(target_path):
                shutil.copy(f_path, target_path)
                total_files_copied_val += 1

    print(f"Processed {machine_clean} (Additional).")

print(f"\n\n ADDITIONAL SPLIT COMPLETE.")
print(f"Total copied to 'train_additional': {total_files_copied_train}")
print(f"Total copied to 'val_additional': {total_files_copied_val}")

In [ ]:
import os
import glob
import shutil
import numpy as np

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DIR = os.environ.get('BASE_DIR', '/content/drive/MyDrive/MasterProject')

# 1. SOURCE PATH: Where the raw TEST data is located
# Structure: .../data/raw/mimii_dg/dev_bearing/bearing/test/*.wav
RAW_DATA_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'mimii_dg')

# 2. DESTINATION PATHS: Where the new splits will go
# Structure: .../data/splits/test/test_train/bearing/*.wav
SPLITS_TEST_DIR = os.path.join(BASE_DIR, 'data', 'splits', 'test')
DEST_TEST_TRAIN = os.path.join(SPLITS_TEST_DIR, 'test_train')
DEST_TEST_VAL = os.path.join(SPLITS_TEST_DIR, 'test_val')

# Machine types definition
MACHINE_TYPES_DEV = ['dev_bearing', 'dev_fan', 'dev_gearbox', 'dev_slider', 'dev_ToyCar', 'dev_ToyTrain', 'dev_valve']
MACHINE_NAMES_CLEAN = [m.replace('dev_', '') for m in MACHINE_TYPES_DEV]

# Reproducibility (Ensures the shuffle is always the same)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Counters for logging
total_copied_train = 0
total_copied_val = 0

print(f"--- STARTING STRATIFIED SPLIT (RAW TEST -> SPLITS) ---")
print(f"Source: {RAW_DATA_DIR} (.../test)")
print(f"Dest Train (90%): {DEST_TEST_TRAIN}")
print(f"Dest Val (10%):   {DEST_TEST_VAL}")

# ==========================================
# MAIN LOOP
# ==========================================
for machine_folder, machine_clean in zip(MACHINE_TYPES_DEV, MACHINE_NAMES_CLEAN):

    # Define Source Directory (The raw 'test' folder)
    SOURCE_DIR = os.path.join(RAW_DATA_DIR, machine_folder, machine_clean, 'test')

    # Define Destination Directories for this specific machine
    current_dest_train = os.path.join(DEST_TEST_TRAIN, machine_clean)
    current_dest_val = os.path.join(DEST_TEST_VAL, machine_clean)

    # Create directories if they don't exist
    os.makedirs(current_dest_train, exist_ok=True)
    os.makedirs(current_dest_val, exist_ok=True)

    # Find all .wav files in the source test folder
    all_wav_files = glob.glob(os.path.join(SOURCE_DIR, "*.wav"))

    if not all_wav_files:
        print(f"⚠️ No files found in {SOURCE_DIR}. Skipping.")
        continue

    print(f"Processing {machine_clean}...")

    # Iterate through Sections (00, 01, 02) to ensure balanced sections
    sections = ["section_00", "section_01", "section_02"]

    for section in sections:
        # Filter files belonging to the current section
        section_files = [f for f in all_wav_files if section in os.path.basename(f)]

        if not section_files:
            continue

        # ------------------------------------------------
        # STRATIFIED CATEGORIZATION
        # We separate files into 4 lists to ensure every type is represented in Val.
        # ------------------------------------------------
        categories = {
            "source_normal":  [f for f in section_files if "source_test_normal" in os.path.basename(f)],
            "target_normal":  [f for f in section_files if "target_test_normal" in os.path.basename(f)],
            "source_anomaly": [f for f in section_files if "source_test_anomaly" in os.path.basename(f)],
            "target_anomaly": [f for f in section_files if "target_test_anomaly" in os.path.basename(f)]
        }

        # Process each category independently
        for cat_name, file_list in categories.items():

            # 1. Shuffle the list randomly
            np.random.shuffle(file_list)

            n_files = len(file_list)
            if n_files == 0:
                continue

            # 2. Calculate Split Index (90% Train / 10% Val)
            # Example: 50 files -> int(45.0) -> index 45
            split_idx = int(n_files * 0.9)

            # 3. Slice the list (Guarantees NO duplicates)
            # Train gets index 0 to 44
            train_subset = file_list[:split_idx]
            # Val gets index 45 to end
            val_subset = file_list[split_idx:]

            # 4. Copy files to 'test_train'
            for f_path in train_subset:
                fname = os.path.basename(f_path)
                dest = os.path.join(current_dest_train, fname)
                # Avoid re-copying if already exists to save time
                if not os.path.exists(dest):
                    shutil.copy(f_path, dest)
                    total_copied_train += 1

            # 5. Copy files to 'test_val'
            for f_path in val_subset:
                fname = os.path.basename(f_path)
                dest = os.path.join(current_dest_val, fname)
                if not os.path.exists(dest):
                    shutil.copy(f_path, dest)
                    total_copied_val += 1

print(f"\n✅ SUCCESS: TEST DATA SPLIT COMPLETE.")
print(f"Total files moved to 'test_train': {total_copied_train}")
print(f"Total files moved to 'test_val':   {total_copied_val}")

--- STARTING STRATIFIED SPLIT (RAW TEST -> SPLITS) ---
Source: /content/drive/MyDrive/MasterProject/data/raw/mimii_dg (.../test)
Dest Train (90%): /content/drive/MyDrive/MasterProject/data/splits/test/test_train
Dest Val (10%):   /content/drive/MyDrive/MasterProject/data/splits/test/test_val
Processing bearing...
Processing fan...
Processing gearbox...
Processing slider...
Processing ToyCar...
Processing ToyTrain...
Processing valve...

✅ SUCCESS: TEST DATA SPLIT COMPLETE.
Total files moved to 'test_train': 3780
Total files moved to 'test_val':   420


In [ ]:
# Copy TEST Data to Splits

# Configuration
BASE_DIR = '/content/drive/MyDrive/MasterProject'

if not os.path.exists(BASE_DIR):
    raise ValueError(f"ERROR: BASE_DIR not found at {BASE_DIR}.")

# Source-Base (raw)
RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'mimii_dg')

# Target-Base (splits)
TARGET_TEST_BASE = os.path.join(BASE_DIR, 'data', 'splits', 'test')

# Maschine-Definition
MACHINE_TYPES_DEV = ['dev_bearing', 'dev_fan', 'dev_gearbox', 'dev_slider', 'dev_ToyCar', 'dev_ToyTrain', 'dev_valve']
MACHINE_NAMES_CLEAN = [m.replace('dev_', '') for m in MACHINE_TYPES_DEV]

# Global Counter
total_files_copied = 0

# Main Loop
for machine_folder, machine_clean in zip(MACHINE_TYPES_DEV, MACHINE_NAMES_CLEAN):

    # Source path: .../mimii_dg/dev_bearing/bearing/test/
    source_test_dir = os.path.join(RAW_DIR, machine_folder, machine_clean, 'test')

    # Target path: .../splits/test/bearing/
    target_test_dir = os.path.join(TARGET_TEST_BASE, machine_clean)
    os.makedirs(target_test_dir, exist_ok=True)

    if not os.path.exists(source_test_dir):
        print(f"Warning: No 'test' folder found for {machine_clean} at {source_test_dir}")
        continue

    # Find all .wav files
    test_files = glob.glob(os.path.join(source_test_dir, "*.wav"))

    if not test_files:
        print(f"Folder exists but is empty: {source_test_dir}")
        continue

    print(f"Found {len(test_files)} test files. Copying...")

    # Copy files
    copied_count = 0
    for f_path in tqdm(test_files, desc=f"Copying {machine_clean}"):
        filename = os.path.basename(f_path)
        target_path = os.path.join(target_test_dir, filename)

        # Only copy if the file does not exist yet (Safety / Time saver)
        if not os.path.exists(target_path):
            shutil.copy(f_path, target_path)
            copied_count += 1

    print(f"Copied {copied_count} files to {target_test_dir}")
    total_files_copied += copied_count

print(f"\n\n TEST DATA COPY COMPLETE. Total copied: {total_files_copied}")

--- Starting Copy: TEST Data ---

=== Processing bearing (Test Data) ===
Found 600 test files. Copying...


Copying bearing:   0%|          | 0/600 [00:00<?, ?it/s]

✅ Copied 600 files to /content/drive/MyDrive/MasterProject/data/splits/test/bearing

=== Processing fan (Test Data) ===
Found 600 test files. Copying...


Copying fan:   0%|          | 0/600 [00:00<?, ?it/s]

✅ Copied 600 files to /content/drive/MyDrive/MasterProject/data/splits/test/fan

=== Processing gearbox (Test Data) ===
Found 600 test files. Copying...


...

✅ Copied 600 files to /content/drive/MyDrive/MasterProject/data/splits/test/ToyCar

=== Processing ToyTrain (Test Data) ===
Found 600 test files. Copying...


Copying ToyTrain:   0%|          | 0/600 [00:00<?, ?it/s]

✅ Copied 600 files to /content/drive/MyDrive/MasterProject/data/splits/test/ToyTrain

=== Processing valve (Test Data) ===
Found 600 test files. Copying...


Copying valve:   0%|          | 0/600 [00:00<?, ?it/s]

✅ Copied 600 files to /content/drive/MyDrive/MasterProject/data/splits/test/valve


✅✅✅ TEST DATA COPY COMPLETE. Total copied: 4200 ✅✅✅


In [ ]:
# ============================================================
# SCRIPT: Bandwidth & Quantizer Configuration Check
# ============================================================

print("--- BANDWIDTH CONFIGURATION CHECK ---")

# Path to the serialized latent file (generated in the previous extraction step)
LATENT_PATH = "/content/drive/MyDrive/MasterProject/outputs/samples/test_float_latents.pth"

if os.path.exists(LATENT_PATH):
    # Load the dictionary containing latents and metadata
    data = torch.load(LATENT_PATH)
    latents = data['latents']

    print(f"Latents Tensor Shape: {latents.shape}")
    # Note: For EnCodec, the shape typically represents [Batch, Channels/Codebooks, Time]

    # Identify the number of active quantizers/codebooks
    # This directly correlates to the selected bit rate (kbps)
    num_codebooks = latents.shape[1]

    print(f"\nNumber of active Codebooks/Channels: {num_codebooks}")

    # Logic based on EnCodec 24kHz model specifications:
    # 32 codebooks = 24.0 kbps (Maximum fidelity)
    # 16 codebooks = 12.0 kbps
    # 8 codebooks  = 6.0 kbps
    # 4 codebooks  = 3.0 kbps
    # 2 codebooks  = 1.5 kbps

    if num_codebooks == 32 or num_codebooks == 128:
        # Note: In your float extraction, 128 channels represent the uncompressed
        # features which provide the highest possible data density for Diffusion.
        print("CONFIRMED: High-Fidelity representation detected (24.0 kbps or raw encoder output).")
    elif num_codebooks == 8:
        print("WARNING: Standard 6.0 kbps detected. This might lose subtle anomaly details.")
    else:
        print(f"Custom configuration: {num_codebooks} codebooks/channels active.")

else:
    print("Error: Latent file not found. Please run the extraction script first.")

--- BANDBREITEN-CHECK ---
Latents Shape: torch.Size([1, 32, 750])

Anzahl Codebooks: 32
✅ BESTÄTIGT: Das ist High-Quality 24.0 kbps!


In [ ]:
# ============================================================
# SCRIPT: Batch Generation of Continuous Float Latents
# ============================================================

# Suppress warnings to maintain a clean execution log
warnings.filterwarnings("ignore")

print("--- RE-GENERATING LATENTS (CONTINUOUS / FLOAT VECTORS) ---")

# 1. Configuration & Path Setup
BASE_DIR           = '/content/drive/MyDrive/MasterProject'
SOURCE_SPLIT_DIR   = os.path.join(BASE_DIR, 'data', 'splits')
TARGET_FEATURE_DIR = os.path.join(BASE_DIR, 'data', 'features', 'encodec')

TARGET_SAMPLE_RATE = 24000
# Define machine categories and dataset partitions
MACHINES = ['bearing', 'fan', 'gearbox', 'slider', 'valve', 'ToyCar', 'ToyTrain']
SPLITS   = ['train', 'val', 'train_additional', 'val_additional', 'test']

# 2. Model Initialization
print("Loading EnCodec model (24kHz)...")
device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    model = EncodecModel.encodec_model_24khz()
    # High-quality bandwidth setting (24.0 kbps)
    model.set_target_bandwidth(24.0)
    model.to(device)
    model.eval() # Set to evaluation mode for consistent feature extraction
    print(f"Model successfully loaded on {device}.")
except ImportError:
    print("Encodec not found! Please run '!pip install encodec'.")
    raise

def process_and_save_continuous(wav_path, save_path):
    """
    Processes raw audio into continuous float latents using the EnCodec encoder.
    Bypasses the quantizer to preserve high-fidelity features for Diffusion training.
    """
    try:
        # A. Load Audio
        wav, sr = torchaudio.load(wav_path)

        # B. Downmix to Mono if stereo detected
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        # C. Resample to 24kHz (EnCodec requirement)
        wav = convert_audio(wav, sr, TARGET_SAMPLE_RATE, model.channels)

        # D. Add Batch Dimension and move to device
        wav = wav.unsqueeze(0).to(device)

        # E. Latent Extraction via Encoder
        # CRITICAL: We use model.encoder() instead of model.encode()
        # model.encode() -> Discrete indices (incorrect for Diffusion training)
        # model.encoder() -> Continuous float vectors [1, 128, T] (correct)
        with torch.no_grad():
            latent_tensor = model.encoder(wav)

        # F. Save as PyTorch Tensor (.pt) to optimize storage and loading speed
        torch.save(latent_tensor.cpu(), save_path)
        return True

    except Exception as e:
        print(f"Error processing {os.path.basename(wav_path)}: {e}")
        return False

# --- MAIN EXTRACTION LOOP ---
total_success = 0
total_skipped = 0

for split in SPLITS:
    current_split_dir = os.path.join(SOURCE_SPLIT_DIR, split)
    if not os.path.exists(current_split_dir):
        # Skip if the source partition folder does not exist
        continue

    for machine in MACHINES:
        # Handle case-insensitive folder searching (important for ToyCar/ToyTrain)
        found_machine_folder = None
        if os.path.exists(current_split_dir):
            for folder in os.listdir(current_split_dir):
                if folder.lower() == machine.lower():
                    found_machine_folder = os.path.join(current_split_dir, folder)
                    break

        if not found_machine_folder:
            continue

        # Define and create output directory: e.g., data/features/encodec/train/bearing
        output_dir = os.path.join(TARGET_FEATURE_DIR, split, machine)
        os.makedirs(output_dir, exist_ok=True)

        # Aggregate all WAV files in the current folder
        wav_files = glob.glob(os.path.join(found_machine_folder, "*.wav"))
        if not wav_files: continue

        print(f"\nProcessing partition: {split}/{machine} ({len(wav_files)} files)...")

        # Process individual files with a nested progress bar
        for wav_path in tqdm(wav_files, desc=f"   Encoding", leave=False):
            filename = os.path.basename(wav_path)
            # Change extension from .wav to .pt for processed features
            save_name = os.path.splitext(filename)[0] + ".pt"
            save_path = os.path.join(output_dir, save_name)

            # Check for existing files to support script resumption
            if os.path.exists(save_path):
                total_skipped += 1
                continue

            if process_and_save_continuous(wav_path, save_path):
                total_success += 1

print("\n" + "="*40)
print(f"EXTRACTION COMPLETE! {total_success} new files generated.")
print(f"Skipped (already exist): {total_skipped}")
print(f"Storage Location: {TARGET_FEATURE_DIR}")

--- 🎵 RE-GENERATING LATENTS (CONTINUOUS / FLOAT) ---
⏳ Lade EnCodec Modell (24kHz)...
✅ Modell geladen auf cuda.

🔵 Processing train/bearing (2729 files)...


   Encoding:   0%|          | 0/2729 [00:00<?, ?it/s]


🔵 Processing train/fan (2730 files)...


   Encoding:   0%|          | 0/2730 [00:00<?, ?it/s]


🔵 Processing train/gearbox (2730 files)...


...


🔵 Processing test/ToyCar (600 files)...


   Encoding:   0%|          | 0/600 [00:00<?, ?it/s]


🔵 Processing test/ToyTrain (600 files)...


   Encoding:   0%|          | 0/600 [00:00<?, ?it/s]


🎉 FERTIG! 0 Dateien neu erstellt.
⏩ Übersprungen: 46199
💾 Gespeichert in: /content/drive/MyDrive/MasterProject/data/features/encodec


In [ ]:
import os
import glob
import torch
import torchaudio
import warnings
from encodec import EncodecModel
from encodec.utils import convert_audio
from tqdm.auto import tqdm

# Suppress warnings to maintain a clean execution log
warnings.filterwarnings("ignore")

print("--- RE-GENERATING LATENTS FOR TEST SPLITS (test_train / test_val) ---")

# 1. Configuration & Path Setup
BASE_DIR           = '/content/drive/MyDrive/MasterProject'
SOURCE_SPLIT_DIR   = os.path.join(BASE_DIR, 'data', 'splits')
TARGET_FEATURE_DIR = os.path.join(BASE_DIR, 'data', 'latents') # Oder 'data/features/encodec', je nach deiner Struktur

TARGET_SAMPLE_RATE = 24000
MACHINES = ['bearing', 'fan', 'gearbox', 'slider', 'valve', 'ToyCar', 'ToyTrain']

# WICHTIG: Hier definieren wir jetzt die Pfade zu den neuen Splits
# Da sie in einem Unterordner "test" liegen, schreiben wir "test/test_train"
SPLITS = ['test/test_train', 'test/test_val']

# 2. Model Initialization
print("Loading EnCodec model (24kHz)...")
device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    model = EncodecModel.encodec_model_24khz()
    # High-quality bandwidth setting (24.0 kbps)
    model.set_target_bandwidth(24.0)
    model.to(device)
    model.eval()
    print(f"Model successfully loaded on {device}.")
except ImportError:
    print("Encodec not found! Please run '!pip install encodec'.")
    raise

def process_and_save_continuous(wav_path, save_path):
    """
    Processes raw audio into continuous float latents using the EnCodec encoder.
    """
    try:
        # A. Load Audio
        wav, sr = torchaudio.load(wav_path)

        # B. Downmix to Mono if stereo detected
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        # C. Resample to 24kHz (EnCodec requirement)
        wav = convert_audio(wav, sr, TARGET_SAMPLE_RATE, model.channels)

        # D. Add Batch Dimension and move to device
        wav = wav.unsqueeze(0).to(device)

        # E. Latent Extraction via Encoder
        # model.encoder() -> Continuous float vectors [1, 128, T]
        with torch.no_grad():
            latent_tensor = model.encoder(wav)

        # F. Save as PyTorch Tensor
        torch.save(latent_tensor.cpu(), save_path)
        return True

    except Exception as e:
        print(f"Error processing {os.path.basename(wav_path)}: {e}")
        return False

# --- MAIN EXTRACTION LOOP ---
total_success = 0
total_skipped = 0

for split in SPLITS:
    # Hier baut er den Pfad: .../data/splits/test/test_train
    current_split_dir = os.path.join(SOURCE_SPLIT_DIR, split)

    if not os.path.exists(current_split_dir):
        print(f"⚠️ Warning: Split directory not found: {current_split_dir}")
        continue

    for machine in MACHINES:
        # Handle case-insensitive folder searching
        found_machine_folder = None
        if os.path.exists(current_split_dir):
            for folder in os.listdir(current_split_dir):
                if folder.lower() == machine.lower():
                    found_machine_folder = os.path.join(current_split_dir, folder)
                    break

        if not found_machine_folder:
            continue

        # Output dir baut sich analog auf: .../data/latents/test/test_train/bearing
        output_dir = os.path.join(TARGET_FEATURE_DIR, split, machine)
        os.makedirs(output_dir, exist_ok=True)

        # Aggregate all WAV files
        wav_files = glob.glob(os.path.join(found_machine_folder, "*.wav"))
        if not wav_files: continue

        print(f"\nProcessing partition: {split} -> {machine} ({len(wav_files)} files)...")

        # Process individual files
        for wav_path in tqdm(wav_files, desc=f"   Encoding", leave=False):
            filename = os.path.basename(wav_path)
            # Change extension from .wav to .pt
            save_name = os.path.splitext(filename)[0] + ".pt"
            save_path = os.path.join(output_dir, save_name)

            # Check for existing files
            if os.path.exists(save_path):
                total_skipped += 1
                continue

            if process_and_save_continuous(wav_path, save_path):
                total_success += 1

print("\n" + "="*40)
print(f"EXTRACTION COMPLETE! {total_success} new files generated.")
print(f"Skipped (already exist): {total_skipped}")
print(f"Storage Location: {TARGET_FEATURE_DIR}")

--- RE-GENERATING LATENTS FOR TEST SPLITS (test_train / test_val) ---
Loading EnCodec model (24kHz)...
Downloading: "https://dl.fbaipublicfiles.com/encodec/v0/encodec_24khz-d7cc33bc.th" to /root/.cache/torch/hub/checkpoints/encodec_24khz-d7cc33bc.th


100%|██████████| 88.9M/88.9M [00:00<00:00, 230MB/s]


Model successfully loaded on cuda.

Processing partition: test/test_train -> bearing (540 files)...


   Encoding:   0%|          | 0/540 [00:00<?, ?it/s]


Processing partition: test/test_train -> fan (540 files)...


...


Processing partition: test/test_val -> ToyCar (60 files)...


   Encoding:   0%|          | 0/60 [00:00<?, ?it/s]


Processing partition: test/test_val -> ToyTrain (60 files)...


   Encoding:   0%|          | 0/60 [00:00<?, ?it/s]


EXTRACTION COMPLETE! 4200 new files generated.
Skipped (already exist): 0
Storage Location: /content/drive/MyDrive/MasterProject/data/latents


In [ ]:
# ============================================================
# SCRIPT: Generate Captions for BEARING - TRAIN & VAL
# Logic: Parsing metadata from filenames into natural language
# ============================================================


print("--- Creating Captions: BEARING - TRAIN & VAL (FROM FILENAMES) ---")

# 1. Configuration
BASE_DIR = '/content/drive/MyDrive/MasterProject'
SPLIT_DIR = os.path.join(BASE_DIR, 'data/splits')

MACHINES = ['bearing']
SECTIONS = ['00', '01', '02']
SPLITS = ['train', 'val', 'train_additional', 'val_additional']

# 2. Prepare Directory Structure
OUTPUT_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', 'bearing')

# Buffer for console examples
example_source_print = None
example_target_print = None

# 3. Caption Generator Function
def generate_caption_from_filename(filename, section):
    """
    Parses the MIMII-DG filename convention to create descriptive captions.
    Extracts velocity, microphone location, and noise type.
    """
    # --- A. PARAMETER EXTRACTION ---

    # Extract Rotation Velocity (e.g., vel_6 -> 6 krpm)
    velocity = "6"
    vel_match = re.search(r'vel_(\d+)', filename)
    if vel_match: velocity = vel_match.group(1)

    # Extract Microphone Location (e.g., loc_A)
    loc = "A"
    loc_match = re.search(r'loc_([A-Z])', filename)
    if loc_match: loc = loc_match.group(1)

    # Extract Factory Noise Type (e.g., f-n_A)
    noise = "A"
    fn_match = re.search(r'f-n_([A-Z])', filename)
    if fn_match: noise = fn_match.group(1)

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename
    is_target = "target" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = "A bearing"

    if is_anomaly:
        caption += " operating"
    else:
        caption += " operating normally"

    # Section 00 logic (Focused on Velocity Shifts)
    if section == '00':
        caption += f" at a rotation velocity of {velocity} krpm"

    # Section 01 logic (Focused on Microphone Location Shifts)
    elif section == '01':
        caption += f" at {velocity} krpm"
        if is_target and loc == "A":
             # Fallback for target domain if location tag is missing from filename
             caption += " recorded at a different microphone position (Target Domain Locations)"
        else:
             caption += f" recorded at microphone location {loc}"

    # Section 02 logic (Focused on Background Noise Type Shifts)
    elif section == '02':
        caption += f" at {velocity} krpm"
        if is_target and noise == "A":
             # Fallback for target domain noise shifts
             caption += " with unseen factory background noise (Target Domain Type)"
        else:
            caption += f" with background factory noise type {noise}"

    # Append Anomaly Details
    if is_anomaly:
        caption += " with anomaly due to eccentricity"

    # Append SNR Constant (MIMII-DG standard for bearing is 12dB)
    caption += ". The sound contains factory noise at 12.0 dB SNR."

    return caption

# --- MAIN PROCESSING LOOP ---
total_files = 0

for machine in MACHINES:
    print(f"\nProcessing {machine}...")

    for split in SPLITS:

        # Categorize output into development or additional folders
        if 'additional' in split:
            sub_folder = 'additional'
        else:
            sub_folder = 'development'

        current_output_dir = os.path.join(OUTPUT_ROOT, sub_folder)
        os.makedirs(current_output_dir, exist_ok=True)

        # Locate source WAV files
        path_with_machine = os.path.join(SPLIT_DIR, split, machine)
        path_direct = os.path.join(SPLIT_DIR, split)

        wav_dir = ""
        if os.path.exists(path_with_machine):
            wav_dir = path_with_machine
        elif os.path.exists(path_direct):
             if glob.glob(os.path.join(path_direct, "*.wav")):
                 wav_dir = path_direct

        if not wav_dir: continue

        all_wavs = glob.glob(os.path.join(wav_dir, "*.wav"))
        if not all_wavs: continue

        print(f"   Split '{split}': {len(all_wavs)} files found.")

        for section in SECTIONS:
            section_str = f"section_{section}"
            sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

            if not sec_files: continue

            captions = {}
            for filepath in sec_files:
                filename = os.path.basename(filepath)
                key = filename.replace('.wav', '')

                # Generate the semantic text
                cap = generate_caption_from_filename(filename, section)
                captions[key] = cap

                # Store examples for log verification
                if "source" in filename and example_source_print is None:
                    example_source_print = f"FILE: {filename}\n   CAPTION: {cap}"

                if "target" in filename and example_target_print is None:
                    example_target_print = f"FILE: {filename}\n   CAPTION: {cap}"

            # Save the gathered captions to a JSON file
            json_filename = f"captions_{machine}_section_{section}_{split}.json"
            json_path = os.path.join(current_output_dir, json_filename)

            with open(json_path, 'w') as f:
                json.dump(captions, f, indent=4)

            print(f"     Saved: {json_filename}")
            total_files += len(captions)

print("\n" + "="*30)
print(f"FINISHED: Total of {total_files} captions created.\n")

print("--- EXAMPLE: SOURCE DOMAIN ---")
if example_source_print:
    print(example_source_print)
else:
    print("   (No source file found)")

print("\n--- EXAMPLE: TARGET DOMAIN ---")
if example_target_print:
    print(example_target_print)
else:
    print("   (No target file found)")

--- Creating Captions: BEARING - TRAIN & VAL (FROM FILENAMES) ---

Processing bearing...
   Split 'train': 2726 files found.
     Saved: captions_bearing_section_00_train.json
     Saved: captions_bearing_section_01_train.json
     Saved: captions_bearing_section_02_train.json
   Split 'val': 273 files found.
     Saved: captions_bearing_section_00_val.json
     Saved: captions_bearing_section_01_val.json
     Saved: captions_bearing_section_02_val.json
   Split 'train_additional': 2730 files found.
   Split 'val_additional': 270 files found.

FINISHED: Total of 2999 captions created.

--- EXAMPLE: SOURCE DOMAIN ---
FILE: section_00_source_train_normal_0815_vel_22.wav
   CAPTION: A bearing operating normally at a rotation velocity of 22 krpm. The sound contains factory noise at 12.0 dB SNR.

--- EXAMPLE: TARGET DOMAIN ---
FILE: section_00_target_train_normal_0007_vel_8.wav
   CAPTION: A bearing operating normally at a rotation velocity of 8 krpm. The sound contains factory noise at 12.

In [ ]:
# ============================================================
# SCRIPT: Generate Captions for BEARING - SECTIONS 03-05
# Logic: Processing Additional Training Data partitions
# ============================================================

import os
import re
import glob
import json

print("--- Creating Captions: BEARING - SECTIONS 03-05 (ADDITIONAL) ---")

# 1. Configuration
BASE_DIR = '/content/drive/MyDrive/MasterProject'
SPLIT_DIR = os.path.join(BASE_DIR, 'data/splits')

MACHINES = ['bearing']
# Target-Domain Sections (Evaluation sections used for additional training)
SECTIONS = ['03', '04', '05']
# Additional data splits
SPLITS = ['train_additional', 'val_additional']

# 2. Prepare Directory Structure
# Target: data/text_captions/bearing/additional
OUTPUT_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', 'bearing')
OUTPUT_SUBFOLDER = 'additional'
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, OUTPUT_SUBFOLDER)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Storage Location: {OUTPUT_DIR}")

# Buffer for console examples
example_source = None
example_target = None

# 3. Caption Generator Function
def generate_caption_from_filename(filename, section):
    """
    Parses filename metadata to create descriptive text.
    Focuses on physical parameters (velocity, location, noise).
    """
    # --- A. PARAMETER EXTRACTION ---

    # Velocity (Default 6 krpm)
    velocity = "6"
    vel_match = re.search(r'vel_(\d+)', filename)
    if vel_match: velocity = vel_match.group(1)

    # Microphone Location (Default A)
    loc = "A"
    loc_match = re.search(r'loc_([A-Z])', filename)
    if loc_match: loc = loc_match.group(1)

    # Noise Type (Default A)
    noise = "A"
    fn_match = re.search(r'f-n_([A-Z])', filename)
    if fn_match: noise = fn_match.group(1)

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = "A bearing"

    if is_anomaly:
        caption += " operating"
    else:
        caption += " operating normally"

    # --- SECTION-SPECIFIC MAPPING ---
    # We describe the physical state rather than using "Target Domain" labels.

    # Section 03: Velocity Shift (corresponding to logic of Section 00)
    if section == '03':
        caption += f" at a rotation velocity of {velocity} krpm"

    # Section 04: Location Shift (corresponding to logic of Section 01)
    elif section == '04':
        caption += f" at {velocity} krpm recorded at microphone location {loc}"

    # Section 05: Noise Shift (corresponding to logic of Section 02)
    elif section == '05':
        caption += f" at {velocity} krpm with background factory noise type {noise}"

    # Anomaly Detail (Eccentricity is the standard for Bearing)
    if is_anomaly:
        caption += " with anomaly due to eccentricity"

    # SNR Constant for Bearing dataset
    caption += ". The sound contains factory noise at 12.0 dB SNR."

    return caption

# --- MAIN PROCESSING LOOP ---
total_files = 0

print(f"\nProcessing {MACHINES[0]}...")

for split_folder in SPLITS:

    # Path Resolution
    path_with_machine = os.path.join(SPLIT_DIR, split_folder, MACHINES[0])
    path_direct = os.path.join(SPLIT_DIR, split_folder)

    wav_dir = ""
    if os.path.exists(path_with_machine):
        wav_dir = path_with_machine
    elif os.path.exists(path_direct):
        if glob.glob(os.path.join(path_direct, "*.wav")):
            wav_dir = path_direct

    if not wav_dir:
        continue

    all_wavs = glob.glob(os.path.join(wav_dir, "*.wav"))
    if not all_wavs: continue

    print(f"   Directory '{split_folder}': {len(all_wavs)} files found.")

    for section in SECTIONS:
        section_str = f"section_{section}"

        # Filtering files by section
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files:
            continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '')

            # Generate semantic caption
            cap = generate_caption_from_filename(filename, section)
            captions[key] = cap

            # Store examples for log verification
            if example_source is None and "source" in filename:
                example_source = f"FILE (Source): {filename}\n   TEXT: {cap}"

            if example_target is None and "target" in filename:
                example_target = f"FILE (Target): {filename}\n   TEXT: {cap}"

        # Save to JSON in the 'additional' subfolder
        json_filename = f"captions_{MACHINES[0]}_section_{section}_{split_folder}.json"
        json_path = os.path.join(OUTPUT_DIR, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"     Saved: {json_filename} ({len(captions)} entries)")
        total_files += len(captions)

print("\n" + "="*30)
print(f"FINISHED: Total of {total_files} Bearing-Captions (Additional) created.\n")

# --- CONSOLE OUTPUT EXAMPLES ---
print("--- GENERATED EXAMPLES ---")
if example_source:
    print(example_source)
    print("-" * 20)
else:
    print("WARNING: No Source-Domain example found.")

if example_target:
    print(example_target)
else:
    print("WARNING: No Target-Domain example found.")

--- Creating Captions: BEARING - SECTIONS 03-05 (ADDITIONAL) ---
Storage Location: /content/drive/MyDrive/MasterProject/data/text_captions/bearing/additional

Processing bearing...
   Directory 'train_additional': 2730 files found.
     Saved: captions_bearing_section_03_train_additional.json (910 entries)
     Saved: captions_bearing_section_04_train_additional.json (910 entries)
     Saved: captions_bearing_section_05_train_additional.json (910 entries)
   Directory 'val_additional': 270 files found.
     Saved: captions_bearing_section_03_val_additional.json (90 entries)
     Saved: captions_bearing_section_04_val_additional.json (90 entries)
     Saved: captions_bearing_section_05_val_additional.json (90 entries)

FINISHED: Total of 3000 Bearing-Captions (Additional) created.

--- GENERATED EXAMPLES ---
FILE (Source): section_03_source_train_normal_0196_vel_9.wav
   TEXT: A bearing operating normally at a rotation velocity of 9 krpm. The sound contains factory noise at 12.0 dB SNR.

In [ ]:
import os
import glob
import re
import json

# ============================================================
# SCRIPT: Generate Captions for BEARING - TEST DATA (SPLIT)
# Logic: Processing 'test_train' and 'test_val' folders separately
# ============================================================

print("--- Creating Captions: BEARING - TEST SPLITS (TRAIN/VAL) ---")

# 1. Configuration
BASE_DIR = '/content/drive/MyDrive/MasterProject'
# Der Hauptordner, wo die beiden neuen Split-Ordner liegen
TEST_SPLIT_ROOT = os.path.join(BASE_DIR, 'data/splits/test')

# Die beiden Unterordner, die wir bearbeiten müssen
SUB_SPLITS = ['test_train', 'test_val']
MACHINE = 'bearing'
SECTIONS = ['00', '01', '02']

# Ziel-Basisordner für Captions
# Struktur wird sein: data/text_captions/bearing/test/test_train/ und .../test_val/
CAPTION_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE, 'test')

# Global variables for console examples (just for checking)
example_source = None
example_target = None

# 2. Caption Generator Function (Bleibt gleich)
def generate_test_caption(filename, section):
    """
    Extracts parameters using regex and builds descriptive text for test samples.
    """
    # --- A. PARAMETER EXTRACTION ---
    velocity = "6"
    vel_match = re.search(r'vel_(\d+)', filename)
    if vel_match: velocity = vel_match.group(1)

    loc = "A"
    loc_match = re.search(r'loc_([A-Z])', filename)
    if loc_match: loc = loc_match.group(1)

    noise = "A"
    fn_match = re.search(r'f-n_([A-Z])', filename)
    if fn_match: noise = fn_match.group(1)

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename
    # Im Testset gibt es oft "inner race", "outer race" etc., aber Baseline unterscheidet meist nur Anomaly
    # Wir machen es hier generisch, es sei denn du willst spezifische Fehler benennen.

    # --- C. TEXT CONSTRUCTION ---
    caption = "A bearing"

    if is_anomaly:
        caption += " operating with anomaly"
    else:
        caption += " operating normally"

    # --- SECTION-SPECIFIC LOGIC ---
    if section == '00':
        caption += f" at a rotation velocity of {velocity} krpm"
    elif section == '01':
        caption += f" at {velocity} krpm recorded at microphone location {loc}"
    elif section == '02':
        caption += f" at {velocity} krpm with background factory noise type {noise}"

    # Anomaly Detail (Optional für Bearing)
    if is_anomaly:
        caption += " due to eccentricity" # Annahme für Baseline, kann angepasst werden

    # SNR Constant
    caption += ". The sound contains factory noise at 12.0 dB SNR."

    return caption

# 3. MAIN LOOP (Durchläuft test_train UND test_val)
for sub_split in SUB_SPLITS:

    # Pfade definieren
    # Input: .../splits/test/test_train/bearing
    input_dir = os.path.join(TEST_SPLIT_ROOT, sub_split, MACHINE)

    # Output: .../text_captions/bearing/test/test_train
    output_dir = os.path.join(CAPTION_ROOT, sub_split)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n📂 Processing Split: {sub_split.upper()}")
    print(f"   Input:  {input_dir}")
    print(f"   Output: {output_dir}")

    if not os.path.exists(input_dir):
        print(f"   ⚠️ Warning: Directory not found: {input_dir}")
        continue

    # Alle WAVs holen
    all_wavs = glob.glob(os.path.join(input_dir, "*.wav"))
    if not all_wavs:
        print("   ⚠️ No WAV files found.")
        continue

    print(f"   Found {len(all_wavs)} audio files.")

    total_files_split = 0

    # Sektions-Loop
    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files:
            continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '') # Key für JSON (ohne Endung)

            # Caption erstellen
            cap = generate_test_caption(filename, section)
            captions[key] = cap

            # Ein Beispiel für die Konsole speichern
            if sub_split == 'test_train': # Nur vom Train-Split Beispiele zeigen
                if example_source is None and "source" in filename:
                    example_source = f"[{sub_split}] SOURCE: {filename}\n   TEXT: {cap}"
                if example_target is None and "target" in filename:
                    example_target = f"[{sub_split}] TARGET: {filename}\n   TEXT: {cap}"

        # Speichern (JSON Name enthält jetzt auch den Split-Namen zur Sicherheit)
        json_filename = f"captions_{MACHINE}_section_{section}_{sub_split}.json"
        json_path = os.path.join(output_dir, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"   ✅ Saved: {json_filename} ({len(captions)} entries)")
        total_files_split += len(captions)

print("\n" + "="*40)
print("FINISHED CAPTION GENERATION.")

# Examples
print("--- GENERATED EXAMPLES ---")
if example_source: print(example_source)
print("-" * 20)
if example_target: print(example_target)

--- Creating Captions: BEARING - TEST SPLITS (TRAIN/VAL) ---

📂 Processing Split: TEST_TRAIN
   Input:  /content/drive/MyDrive/MasterProject/data/splits/test/test_train/bearing
   Output: /content/drive/MyDrive/MasterProject/data/text_captions/bearing/test/test_train
   Found 540 audio files.
   ✅ Saved: captions_bearing_section_00_test_train.json (180 entries)
   ✅ Saved: captions_bearing_section_01_test_train.json (180 entries)
   ✅ Saved: captions_bearing_section_02_test_train.json (180 entries)

📂 Processing Split: TEST_VAL
   Input:  /content/drive/MyDrive/MasterProject/data/splits/test/test_val/bearing
   Output: /content/drive/MyDrive/MasterProject/data/text_captions/bearing/test/test_val
   Found 60 audio files.
   ✅ Saved: captions_bearing_section_00_test_val.json (20 entries)
   ✅ Saved: captions_bearing_section_01_test_val.json (20 entries)
   ✅ Saved: captions_bearing_section_02_test_val.json (20 entries)

FINISHED CAPTION GENERATION.
--- GENERATED EXAMPLES ---
[test_train]

In [ ]:
import os
import re
import glob
import json

# ============================================================
# SCRIPT: Generate Captions for FAN - TEST DATA (SPLIT)
# Logic: Processing 'test_train' and 'test_val' folders separately
#        Retains Fan-specific metadata parsing (SNR, Damage Types)
# ============================================================

print("--- Creating Captions: FAN - TEST SPLITS (TRAIN/VAL) ---")

# 1. Configuration
BASE_DIR = '/content/drive/MyDrive/MasterProject'

# Root of the split audio files
TEST_SPLIT_ROOT = os.path.join(BASE_DIR, 'data/splits/test')

# The two sub-splits to process
SUB_SPLITS = ['test_train', 'test_val']
MACHINE = 'fan'
SECTIONS = ['00', '01', '02']

# Target Base Directory for Captions
# Structure will be: data/text_captions/fan/test/test_train/ etc.
CAPTION_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE, 'test')

# Mapping for Noise Levels specific to Fan Section 02
SNR_MAPPING = {
    'L1': '3',
    'L2': '-9',
    'L3': '-3',
    'L4': '-15'
}

# Global variables for console examples
example_source = None
example_target = None

# 2. Caption Generator Function (Fan-Specific)
def generate_test_caption(filename, section):
    """
    Parses Fan metadata: Machine Noise (m-n), Factory Noise (f-n), and Noise Level (n-lv).
    Also detects specific damage keywords if present in the filename.
    """
    # --- A. PARAMETER EXTRACTION (REGEX) ---

    # 1. Machine Noise Index (e.g., m-n_W)
    m_noise = "W"
    mn_match = re.search(r'm-n_([A-Z])', filename)
    if mn_match: m_noise = mn_match.group(1)

    # 2. Factory Noise Index (e.g., f-n_A)
    f_noise = "A"
    fn_match = re.search(r'f-n_([A-Z])', filename)
    if fn_match: f_noise = fn_match.group(1)

    # 3. Noise Level Code (e.g., n-lv_L1)
    noise_level_code = "L1"
    nlv_match = re.search(r'n-lv_([A-Z0-9]+)', filename)
    if nlv_match: noise_level_code = nlv_match.group(1)

    # Retrieve actual dB value from mapping
    snr_db = SNR_MAPPING.get(noise_level_code, "3")

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = "A fan"

    if is_anomaly:
        caption += " operating abnormally"
        # Incorporate specific damage types if detected in filename
        # (These keywords are specific to the MIMII Fan dataset)
        if "wing" in filename.lower(): caption += " due to wing damage"
        elif "clog" in filename.lower(): caption += " due to clogging"
        elif "unbal" in filename.lower(): caption += " due to unbalanced load"
        elif "volt" in filename.lower(): caption += " due to over voltage"
    else:
        caption += " operating normally"

    # --- SECTION-SPECIFIC LOGIC (Based on MIMII-DG Fan Specifications) ---

    # Section 00: Machine Noise Variation -> Fixed SNR -6.0 dB
    if section == '00':
        caption += f" with machine sound index {m_noise}"
        caption += ". The sound contains factory noise at -6.0 dB SNR."

    # Section 01: Factory Noise Variation -> Fixed SNR -12.0 dB
    elif section == '01':
        caption += f" with background factory noise index {f_noise}"
        caption += ". The sound contains factory noise at -12.0 dB SNR."

    # Section 02: Noise Level Variation -> Variable SNR
    elif section == '02':
        caption += f". The sound contains factory noise at {snr_db} dB SNR."

    return caption

# 3. MAIN PROCESSING LOOP (Iterates through splits)
for sub_split in SUB_SPLITS:

    # Input Path: .../splits/test/test_train/fan
    input_dir = os.path.join(TEST_SPLIT_ROOT, sub_split, MACHINE)

    # Output Path: .../text_captions/fan/test/test_train
    output_dir = os.path.join(CAPTION_ROOT, sub_split)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n📂 Processing Split: {sub_split.upper()}")
    print(f"   Input:  {input_dir}")
    print(f"   Output: {output_dir}")

    if not os.path.exists(input_dir):
        print(f"   ⚠️ Warning: Directory not found: {input_dir}")
        continue

    # Find all .wav files
    all_wavs = glob.glob(os.path.join(input_dir, "*.wav"))

    if not all_wavs:
        print("   ⚠️ No WAV files found.")
        continue

    print(f"   Found {len(all_wavs)} audio files.")

    total_files_split = 0

    # Process by Section
    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '') # JSON Key

            # Generate semantic text
            cap = generate_test_caption(filename, section)
            captions[key] = cap

            # Track examples for verification (Only from train split to keep console clean)
            if sub_split == 'test_train':
                if example_source is None and "source" in filename:
                    example_source = f"[{sub_split}] SOURCE: {filename}\n   TEXT: {cap}"
                if example_target is None and "target" in filename:
                    example_target = f"[{sub_split}] TARGET: {filename}\n   TEXT: {cap}"

        # Save JSON (Filename includes split name)
        json_filename = f"captions_{MACHINE}_section_{section}_{sub_split}.json"
        json_path = os.path.join(output_dir, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"   ✅ Saved: {json_filename} ({len(captions)} entries)")
        total_files_split += len(captions)

print("\n" + "="*40)
print("FINISHED FAN CAPTION GENERATION.")

# --- CONSOLE OUTPUT EXAMPLES ---
print("--- GENERATED EXAMPLES ---")
if example_source:
    print(example_source)
    print("-" * 20)
if example_target:
    print(example_target)
else:
    print("WARNING: No Target domain example found.")

--- Creating Captions: FAN - TEST SPLITS (TRAIN/VAL) ---

📂 Processing Split: TEST_TRAIN
   Input:  /content/drive/MyDrive/MasterProject/data/splits/test/test_train/fan
   Output: /content/drive/MyDrive/MasterProject/data/text_captions/fan/test/test_train
   Found 540 audio files.
   ✅ Saved: captions_fan_section_00_test_train.json (180 entries)
   ✅ Saved: captions_fan_section_01_test_train.json (180 entries)
   ✅ Saved: captions_fan_section_02_test_train.json (180 entries)

📂 Processing Split: TEST_VAL
   Input:  /content/drive/MyDrive/MasterProject/data/splits/test/test_val/fan
   Output: /content/drive/MyDrive/MasterProject/data/text_captions/fan/test/test_val
   Found 60 audio files.
   ✅ Saved: captions_fan_section_00_test_val.json (20 entries)
   ✅ Saved: captions_fan_section_01_test_val.json (20 entries)
   ✅ Saved: captions_fan_section_02_test_val.json (20 entries)

FINISHED FAN CAPTION GENERATION.
--- GENERATED EXAMPLES ---
[test_train] SOURCE: section_00_source_test_normal_0

In [ ]:
# ============================================================
# SCRIPT: Generate Captions for FAN - SECTIONS 03-05
# Logic: Processing Additional Training Data (Target Domain partitions)
# ============================================================


print("--- Creating Captions: FAN - SECTIONS 03-05 (ADDITIONAL / CLEAN) ---")

# 1. Configuration
BASE_DIR = '/content/drive/MyDrive/MasterProject'
SPLIT_DIR = os.path.join(BASE_DIR, 'data/splits')

MACHINE = 'fan'
# Evaluation sections used for Domain Generalization training
SECTIONS = ['03', '04', '05']
SPLITS = ['train_additional', 'val_additional']

# 2. Prepare Directory Structure
# Target: data/text_captions/fan/additional
OUTPUT_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE)
OUTPUT_SUBFOLDER = 'additional'
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, OUTPUT_SUBFOLDER)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Storage Location: {OUTPUT_DIR}")

# Mapping for Noise Levels based on MIMII-DG specifications
SNR_MAPPING = {
    'L1': '3',
    'L2': '-9',
    'L3': '-3',
    'L4': '-15'
}

# 3. Caption Generator Function
def generate_caption_from_filename(filename, section):
    """
    Extracts fan-specific attributes from filename metadata.
    Maps machine noise, factory noise, and SNR levels to descriptive text.
    """
    # --- A. PARAMETER EXTRACTION (REGEX) ---

    # Machine Noise Index (e.g., m-n_W)
    m_noise = "W"
    mn_match = re.search(r'm-n_([A-Z])', filename)
    if mn_match: m_noise = mn_match.group(1)

    # Factory Noise Index (e.g., f-n_A)
    f_noise = "A"
    fn_match = re.search(r'f-n_([A-Z])', filename)
    if fn_match: f_noise = fn_match.group(1)

    # Noise Level Code (e.g., n-lv_L1)
    noise_level_code = "L1"
    nlv_match = re.search(r'n-lv_([A-Z0-9]+)', filename)
    if nlv_match: noise_level_code = nlv_match.group(1)

    # Retrieve SNR in decibels
    snr_db = SNR_MAPPING.get(noise_level_code, "3")

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = "A fan"

    if is_anomaly:
        caption += " operating abnormally"
    else:
        caption += " operating normally"

    # --- SECTION-SPECIFIC MAPPING ---

    # Section 03: Machine Noise target domain shift (Fixed -6.0 dB SNR)
    if section == '03':
        caption += f" with machine sound index {m_noise}"
        caption += ". The sound contains factory noise at -6.0 dB SNR."

    # Section 04: Factory Noise target domain shift (Fixed -12.0 dB SNR)
    elif section == '04':
        caption += f" with background factory noise index {f_noise}"
        caption += ". The sound contains factory noise at -12.0 dB SNR."

    # Section 05: SNR level shift (Variable SNR)
    elif section == '05':
        caption += f". The sound contains factory noise at {snr_db} dB SNR."

    return caption

# --- MAIN PROCESSING LOOP ---
total_files = 0
example_print = None

print(f"\nProcessing {MACHINE}...")

for split_folder in SPLITS:

    # Path Resolution
    path_with_machine = os.path.join(SPLIT_DIR, split_folder, MACHINE)
    path_direct = os.path.join(SPLIT_DIR, split_folder)

    wav_dir = ""
    if os.path.exists(path_with_machine):
        wav_dir = path_with_machine
    elif os.path.exists(path_direct):
        if glob.glob(os.path.join(path_direct, "*.wav")):
            wav_dir = path_direct

    if not wav_dir:
        print(f"   Warning: Directory not found: {split_folder}")
        continue

    all_wavs = glob.glob(os.path.join(wav_dir, "*.wav"))
    if not all_wavs: continue

    print(f"   Directory '{split_folder}': {len(all_wavs)} files found.")

    for section in SECTIONS:
        section_str = f"section_{section}"

        # Filtering by section ID
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '')

            # Generate semantic text
            cap = generate_caption_from_filename(filename, section)
            captions[key] = cap

            # Buffer an example for the final report
            if example_print is None:
                example_print = f"FILE: {filename}\n   TEXT: {cap}"

        # Persist results to JSON
        json_filename = f"captions_{MACHINE}_section_{section}_{split_folder}.json"
        json_path = os.path.join(OUTPUT_DIR, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"     Saved: {json_filename} ({len(captions)} entries)")
        total_files += len(captions)

print("\n" + "="*40)
print(f"FINISHED: Total of {total_files} Fan Captions (Additional) created.\n")

if example_print:
    print("--- GENERATED EXAMPLE ---")
    print(example_print)

--- Creating Captions: FAN - SECTIONS 03-05 (ADDITIONAL / CLEAN) ---
Storage Location: /content/drive/MyDrive/MasterProject/data/text_captions/fan/additional

Processing fan...
   Directory 'train_additional': 2730 files found.
     Saved: captions_fan_section_03_train_additional.json (910 entries)
     Saved: captions_fan_section_04_train_additional.json (910 entries)
     Saved: captions_fan_section_05_train_additional.json (910 entries)
   Directory 'val_additional': 270 files found.
     Saved: captions_fan_section_03_val_additional.json (90 entries)
     Saved: captions_fan_section_04_val_additional.json (90 entries)
     Saved: captions_fan_section_05_val_additional.json (90 entries)

FINISHED: Total of 3000 Fan Captions (Additional) created.

--- GENERATED EXAMPLE ---
FILE: section_03_source_train_normal_0509_m-n_X.wav
   TEXT: A fan operating normally with machine sound index X. The sound contains factory noise at -6.0 dB SNR.


In [ ]:
# ============================================================
# SCRIPT: Generate Captions for FAN - TRAIN & VAL
# Logic: Processing Development Data partitions (Sections 00-02)
# ============================================================

print("--- Creating Captions: FAN - TRAIN & VAL (DEVELOPMENT / CLEAN) ---")

# 1. Configuration
BASE_DIR = '/content/drive/MyDrive/MasterProject'
SPLIT_DIR = os.path.join(BASE_DIR, 'data/splits')

MACHINE = 'fan'
SECTIONS = ['00', '01', '02']
SPLITS = ['train', 'val']

# 2. Prepare Directory Structure
# Target path: data/text_captions/fan/development
OUTPUT_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE)
OUTPUT_SUBFOLDER = 'development'
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, OUTPUT_SUBFOLDER)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Storage Location: {OUTPUT_DIR}")

# Mapping for Noise Levels specific to Fan Section 02
SNR_MAPPING = {
    'L1': '3',
    'L2': '-9',
    'L3': '-3',
    'L4': '-15'
}

# Global variables for console examples
example_print = None

# 3. Caption Generator Function
def generate_caption_from_filename(filename, section):
    """
    Parses Fan-specific attributes (machine noise index, factory noise, SNR)
    and constructs a semantic natural language caption.
    """
    # --- A. PARAMETER EXTRACTION (REGEX) ---

    # 1. Machine Noise Index (e.g., m-n_W)
    m_noise = "W"
    mn_match = re.search(r'm-n_([A-Z])', filename)
    if mn_match: m_noise = mn_match.group(1)

    # 2. Factory Noise Index (e.g., f-n_A)
    f_noise = "A"
    fn_match = re.search(r'f-n_([A-Z])', filename)
    if fn_match: f_noise = fn_match.group(1)

    # 3. Noise Level Code (e.g., n-lv_L1)
    noise_level_code = "L1"
    nlv_match = re.search(r'n-lv_([A-Z0-9]+)', filename)
    if nlv_match: noise_level_code = nlv_match.group(1)

    # Map code to decibel value
    snr_db = SNR_MAPPING.get(noise_level_code, "3")

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = "A fan"

    if is_anomaly:
        caption += " operating abnormally"
    else:
        caption += " operating normally"

    # --- SECTION-SPECIFIC LOGIC ---

    # Section 00: Machine Noise Variation -> Fixed SNR -6.0 dB
    if section == '00':
        caption += f" with machine sound index {m_noise}"
        caption += ". The sound contains factory noise at -6.0 dB SNR."

    # Section 01: Factory Noise Variation -> Fixed SNR -12.0 dB
    elif section == '01':
        caption += f" with background factory noise index {f_noise}"
        caption += ". The sound contains factory noise at -12.0 dB SNR."

    # Section 02: Noise Level Variation -> Variable SNR
    elif section == '02':
        # SNR is the primary varying information in this section
        caption += f". The sound contains factory noise at {snr_db} dB SNR."

    return caption

# --- MAIN PROCESSING LOOP ---
total_files = 0

print(f"\nProcessing {MACHINE}...")

for split in SPLITS:
    # Resolve directory paths
    path_with_machine = os.path.join(SPLIT_DIR, split, MACHINE)
    path_direct = os.path.join(SPLIT_DIR, split)

    wav_dir = ""
    if os.path.exists(path_with_machine):
        wav_dir = path_with_machine
    elif os.path.exists(path_direct):
         if glob.glob(os.path.join(path_direct, "*.wav")):
             wav_dir = path_direct

    if not wav_dir:
        print(f"   Warning: Directory not found: {split}")
        continue

    # Gather all WAV files
    all_wavs = glob.glob(os.path.join(wav_dir, "*.wav"))

    if not all_wavs: continue

    print(f"   Split '{split}': {len(all_wavs)} files found.")

    for section in SECTIONS:
        section_str = f"section_{section}"

        # Filter files belonging to current section
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        # Generate dictionary of captions
        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '')

            # Construct semantic caption
            cap = generate_caption_from_filename(filename, section)
            captions[key] = cap

            # Buffer example for final status report
            if example_print is None:
                example_print = f"FILE: {filename}\n   TEXT: {cap}"

        # Save to JSON in the 'development' subfolder
        json_filename = f"captions_{MACHINE}_section_{section}_{split}.json"
        json_path = os.path.join(OUTPUT_DIR, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"     Saved: {json_filename} ({len(captions)} entries)")
        total_files += len(captions)

print("\n" + "="*40)
print(f"FINISHED: Total of {total_files} Fan captions (Train/Val) created.\n")

if example_print:
    print("--- GENERATED EXAMPLE ---")
    print(example_print)

--- Creating Captions: FAN - TRAIN & VAL (DEVELOPMENT / CLEAN) ---
Storage Location: /content/drive/MyDrive/MasterProject/data/text_captions/fan/development

Processing fan...
   Split 'train': 2727 files found.
     Saved: captions_fan_section_00_train.json (909 entries)
     Saved: captions_fan_section_01_train.json (909 entries)
     Saved: captions_fan_section_02_train.json (909 entries)
   Split 'val': 273 files found.
     Saved: captions_fan_section_00_val.json (91 entries)
     Saved: captions_fan_section_01_val.json (91 entries)
     Saved: captions_fan_section_02_val.json (91 entries)

FINISHED: Total of 3000 Fan captions (Train/Val) created.

--- GENERATED EXAMPLE ---
FILE: section_00_source_train_normal_0247_m-n_X.wav
   TEXT: A fan operating normally with machine sound index X. The sound contains factory noise at -6.0 dB SNR.


In [ ]:
# ============================================================
# SCRIPT: Generate Captions for GEARBOX - TRAIN & VAL
# Logic: Parsing Gearbox-specific metadata (Voltage, Load Weight, Machine ID)
# ============================================================

print("--- Creating Captions: GEARBOX - TRAIN & VAL (DEVELOPMENT) ---")

# 1. Configuration
BASE_DIR = '/content/drive/MyDrive/MasterProject'
SPLIT_DIR = os.path.join(BASE_DIR, 'data/splits')

MACHINE = 'gearbox'
SECTIONS = ['00', '01', '02']
SPLITS = ['train', 'val']

# 2. Prepare Directory Structure
# Target path: data/text_captions/gearbox/development
OUTPUT_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE)
OUTPUT_SUBFOLDER = 'development'
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, OUTPUT_SUBFOLDER)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Storage Location: {OUTPUT_DIR}")

# Dictionary to store examples for final logging
examples_per_section = {
    '00': {'source': None, 'target': None},
    '01': {'source': None, 'target': None},
    '02': {'source': None, 'target': None}
}

# 3. Caption Generator Function
def generate_caption_from_filename(filename, section):
    """
    Parses Gearbox metadata: Operating Voltage, Load Weight, and Machine ID.
    Constructs a semantic caption describing the physical state and noise conditions.
    """
    # --- A. PARAMETER EXTRACTION (REGEX) ---

    # Extract Operating Voltage (Section 00 focus)
    voltage = "3.0"
    volt_match = re.search(r'volt_([0-9.]+)', filename, re.IGNORECASE)
    if volt_match: voltage = volt_match.group(1)

    # Extract Load Weight in grams (Section 01 focus)
    weight = "0"
    wt_match = re.search(r'wt_([0-9]+)', filename, re.IGNORECASE)
    if wt_match: weight = wt_match.group(1)

    # Extract Physical Machine ID (Section 02 focus)
    machine_id = "00"
    id_match = re.search(r'id_([0-9]+)', filename, re.IGNORECASE)
    if id_match: machine_id = id_match.group(1)

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = "A gearbox"

    if is_anomaly:
        caption += " operating abnormally"
        # Specific damage type detection based on keywords
        if "gear" in filename.lower(): caption += " due to gear damage"
        elif "volt" in filename.lower() and "over" in filename.lower(): caption += " due to over voltage"
    else:
        caption += " operating normally"

    # --- D. SECTION-SPECIFIC LOGIC (MIMII-DG Gearbox Specs) ---

    # Section 00: Voltage Variation -> Fixed SNR -6.0 dB
    if section == '00':
        caption += f" at an operating voltage of {voltage}V"
        caption += ". The sound contains factory noise at -6.0 dB SNR."

    # Section 01: Weight/Load Variation -> Fixed SNR -12.0 dB
    elif section == '01':
        caption += f" with a load weight of {weight}g"
        caption += ". The sound contains factory noise at -12.0 dB SNR."

    # Section 02: Machine ID/Serial Variation -> Fixed SNR -12.0 dB
    elif section == '02':
        caption += f" utilizing physical machine ID {machine_id}"
        caption += ". The sound contains factory noise at -12.0 dB SNR."

    return caption

# --- MAIN PROCESSING LOOP ---
total_files = 0

print(f"\nProcessing {MACHINE}...")

for split in SPLITS:
    # Resolve directory paths for current split
    path_with_machine = os.path.join(SPLIT_DIR, split, MACHINE)
    path_direct = os.path.join(SPLIT_DIR, split)

    wav_dir = ""
    if os.path.exists(path_with_machine):
        wav_dir = path_with_machine
    elif os.path.exists(path_direct):
         if glob.glob(os.path.join(path_direct, "*.wav")):
             wav_dir = path_direct

    if not wav_dir:
        print(f"   Warning: Directory not found: {split}")
        continue

    all_wavs = glob.glob(os.path.join(wav_dir, "*.wav"))
    if not all_wavs: continue

    print(f"   Split '{split}': {len(all_wavs)} files found.")

    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '')

            # Generate semantic text
            cap = generate_caption_from_filename(filename, section)
            captions[key] = cap

            # Collect examples for verification purposes
            if "source" in filename and examples_per_section[section]['source'] is None:
                examples_per_section[section]['source'] = f"FILE: {filename}\n      TEXT: {cap}"

            if "target" in filename and examples_per_section[section]['target'] is None:
                examples_per_section[section]['target'] = f"FILE: {filename}\n      TEXT: {cap}"

        # Persist to JSON
        json_filename = f"captions_{MACHINE}_section_{section}_{split}.json"
        json_path = os.path.join(OUTPUT_DIR, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"     Saved: {json_filename} ({len(captions)} entries)")
        total_files += len(captions)

print("\n" + "="*40)
print(f"FINISHED: Total of {total_files} Gearbox captions (Train/Val) created.\n")

# --- CONSOLE OUTPUT EXAMPLES ---
print("--- GENERATED EXAMPLES PER SECTION ---")
for sec in SECTIONS:
    print(f"\nSECTION {sec}:")

    ex_src = examples_per_section[sec]['source']
    if ex_src:
        print(f"   [Source Domain]:\n      {ex_src}")
    else:
        print("   [Source Domain]: No files found.")

    ex_tgt = examples_per_section[sec]['target']
    if ex_tgt:
        print(f"   [Target Domain]:\n      {ex_tgt}")
    else:
        print("   [Target Domain]: No files found (common in development sets).")

--- Creating Captions: GEARBOX - TRAIN & VAL (DEVELOPMENT) ---
Storage Location: /content/drive/MyDrive/MasterProject/data/text_captions/gearbox/development

Processing gearbox...
   Split 'train': 2727 files found.
     Saved: captions_gearbox_section_00_train.json (909 entries)
     Saved: captions_gearbox_section_01_train.json (909 entries)
     Saved: captions_gearbox_section_02_train.json (909 entries)
   Split 'val': 273 files found.
     Saved: captions_gearbox_section_00_val.json (91 entries)
     Saved: captions_gearbox_section_01_val.json (91 entries)
     Saved: captions_gearbox_section_02_val.json (91 entries)

FINISHED: Total of 3000 Gearbox captions (Train/Val) created.

--- GENERATED EXAMPLES PER SECTION ---

SECTION 00:
   [Source Domain]:
      FILE: section_00_source_train_normal_0483_volt_1.0.wav
      TEXT: A gearbox operating normally at an operating voltage of 1.0.V. The sound contains factory noise at -6.0 dB SNR.
   [Target Domain]:
      FILE: section_00_target

In [ ]:
# ============================================================
# SCRIPT: Generate Captions for GEARBOX - ADDITIONAL (03-05)
# Logic: Processing Target Domain partitions for Gearbox
# ============================================================

print("--- Creating Captions: GEARBOX - ADDITIONAL (03-05) WITH EXAMPLES ---")

# 1. Configuration
BASE_DIR = '/content/drive/MyDrive/MasterProject'
SPLIT_DIR = os.path.join(BASE_DIR, 'data/splits')

MACHINE = 'gearbox'
# Target-domain sections used for additional training/generalization
SECTIONS = ['03', '04', '05']
SPLITS = ['train_additional', 'val_additional']

# 2. Prepare Directory Structure
# Target path: data/text_captions/gearbox/additional
OUTPUT_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE)
OUTPUT_SUBFOLDER = 'additional'
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, OUTPUT_SUBFOLDER)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Storage Location: {OUTPUT_DIR}")

# Dictionary to store examples for each section for verification
examples_per_section = {
    '03': {'source': None, 'target': None},
    '04': {'source': None, 'target': None},
    '05': {'source': None, 'target': None}
}

# 3. Caption Generator Function
def generate_caption_from_filename(filename, section):
    """
    Extracts physical parameters from gearbox filenames and constructs semantic captions.
    Covers shifts in voltage, load weight, and machine hardware ID.
    """
    # --- A. PARAMETER EXTRACTION (REGEX) ---

    # Voltage (Relevant for Section 03 shift)
    voltage = "3.0"
    volt_match = re.search(r'volt_([0-9.]+)', filename, re.IGNORECASE)
    if volt_match: voltage = volt_match.group(1)

    # Weight (Relevant for Section 04 shift)
    weight = "0"
    wt_match = re.search(r'wt_([0-9]+)', filename, re.IGNORECASE)
    if wt_match: weight = wt_match.group(1)

    # Machine ID (Relevant for Section 05 shift)
    machine_id = "00"
    id_match = re.search(r'id_([0-9]+)', filename, re.IGNORECASE)
    if id_match: machine_id = id_match.group(1)

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = "A gearbox"

    if is_anomaly:
        caption += " operating abnormally"
        # Optional: Specific fault detection keywords
        if "gear" in filename.lower(): caption += " due to gear damage"
        elif "volt" in filename.lower() and "over" in filename.lower(): caption += " due to over voltage"
    else:
        caption += " operating normally"

    # --- D. SECTION-SPECIFIC LOGIC (TARGET DOMAINS) ---

    # Section 03 (Voltage Target Domain) -> Fixed -6.0 dB SNR
    if section == '03':
        caption += f" at an operating voltage of {voltage}V"
        caption += ". The sound contains factory noise at -6.0 dB SNR."

    # Section 04 (Weight Target Domain) -> Fixed -12.0 dB SNR
    elif section == '04':
        caption += f" with a load weight of {weight}g"
        caption += ". The sound contains factory noise at -12.0 dB SNR."

    # Section 05 (Machine ID Target Domain) -> Fixed -12.0 dB SNR
    elif section == '05':
        caption += f" utilizing physical machine ID {machine_id}"
        caption += ". The sound contains factory noise at -12.0 dB SNR."

    return caption

# --- MAIN PROCESSING LOOP ---
total_files = 0

print(f"\nProcessing {MACHINE}...")

for split_folder in SPLITS:

    # Path Resolution
    path_with_machine = os.path.join(SPLIT_DIR, split_folder, MACHINE)
    path_direct = os.path.join(SPLIT_DIR, split_folder)

    wav_dir = ""
    if os.path.exists(path_with_machine):
        wav_dir = path_with_machine
    elif os.path.exists(path_direct):
         if glob.glob(os.path.join(path_direct, "*.wav")):
             wav_dir = path_direct

    if not wav_dir:
        print(f"   Warning: Directory not found: {split_folder}")
        continue

    all_wavs = glob.glob(os.path.join(wav_dir, "*.wav"))
    if not all_wavs: continue

    print(f"   Directory '{split_folder}': {len(all_wavs)} files found.")

    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '')

            # Generate semantic caption
            cap = generate_caption_from_filename(filename, section)
            captions[key] = cap

            # Collect examples for log verification
            if "source" in filename and examples_per_section[section]['source'] is None:
                examples_per_section[section]['source'] = f"FILE: {filename}\n      TEXT: {cap}"

            if "target" in filename and examples_per_section[section]['target'] is None:
                examples_per_section[section]['target'] = f"FILE: {filename}\n      TEXT: {cap}"

        # Persist to JSON
        json_filename = f"captions_{MACHINE}_section_{section}_{split_folder}.json"
        json_path = os.path.join(OUTPUT_DIR, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"     Saved: {json_filename} ({len(captions)} entries)")
        total_files += len(captions)

print("\n" + "="*40)
print(f"FINISHED: Total of {total_files} Gearbox captions (Additional) created.\n")

# --- CONSOLE OUTPUT EXAMPLES ---
print("--- GENERATED EXAMPLES PER SECTION ---")
for sec in SECTIONS:
    print(f"\nSECTION {sec}:")

    ex_src = examples_per_section[sec]['source']
    if ex_src:
        print(f"   [Source Domain]:\n      {ex_src}")
    else:
        print("   [Source Domain]: No files found.")

    ex_tgt = examples_per_section[sec]['target']
    if ex_tgt:
        print(f"   [Target Domain]:\n      {ex_tgt}")
    else:
        print("   [Target Domain]: No files found.")

--- Creating Captions: GEARBOX - ADDITIONAL (03-05) WITH EXAMPLES ---
Storage Location: /content/drive/MyDrive/MasterProject/data/text_captions/gearbox/additional

Processing gearbox...
   Directory 'train_additional': 2730 files found.
     Saved: captions_gearbox_section_03_train_additional.json (910 entries)
     Saved: captions_gearbox_section_04_train_additional.json (910 entries)
     Saved: captions_gearbox_section_05_train_additional.json (910 entries)
   Directory 'val_additional': 270 files found.
     Saved: captions_gearbox_section_03_val_additional.json (90 entries)
     Saved: captions_gearbox_section_04_val_additional.json (90 entries)
     Saved: captions_gearbox_section_05_val_additional.json (90 entries)

FINISHED: Total of 3000 Gearbox captions (Additional) created.

--- GENERATED EXAMPLES PER SECTION ---

SECTION 03:
   [Source Domain]:
      FILE: section_03_source_train_normal_0871_volt_1.0.wav
      TEXT: A gearbox operating normally at an operating voltage of 1.

In [ ]:
import os
import re
import glob
import json

# ============================================================
# SCRIPT: Generate Captions for GEARBOX - TEST DATA (SPLIT)
# Logic: Processing 'test_train' and 'test_val' folders separately
#        Extracts Voltage, Load Weight, and Machine ID
# ============================================================

print("--- Creating Captions: GEARBOX - TEST SPLITS (TRAIN/VAL) ---")

# 1. Configuration
BASE_DIR = '/content/drive/MyDrive/MasterProject'

# Root of the split audio files
TEST_SPLIT_ROOT = os.path.join(BASE_DIR, 'data/splits/test')

# The two sub-splits to process
SUB_SPLITS = ['test_train', 'test_val']
MACHINE = 'gearbox'
SECTIONS = ['00', '01', '02']

# Target Base Directory for Captions
# Structure will be: data/text_captions/gearbox/test/test_train/ etc.
CAPTION_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE, 'test')

# Global variables for console examples
example_source = None
example_target = None

# 2. Caption Generator Function (Gearbox-Specific)
def generate_test_caption(filename, section):
    """
    Parses Gearbox test metadata: Operating Voltage, Load Weight, and Machine ID.
    Constructs a semantic caption describing the physical state and noise conditions.
    """
    # --- A. PARAMETER EXTRACTION (REGEX) ---

    # Extract Operating Voltage (Section 00)
    voltage = "3.0"
    volt_match = re.search(r'volt_([0-9.]+)', filename, re.IGNORECASE)
    if volt_match: voltage = volt_match.group(1)

    # Extract Load Weight in grams (Section 01)
    weight = "0"
    wt_match = re.search(r'wt_([0-9]+)', filename, re.IGNORECASE)
    if wt_match: weight = wt_match.group(1)

    # Extract Physical Machine ID (Section 02)
    machine_id = "00"
    id_match = re.search(r'id_([0-9]+)', filename, re.IGNORECASE)
    if id_match: machine_id = id_match.group(1)

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = "A gearbox"

    if is_anomaly:
        caption += " operating abnormally"
        # Specific fault detection keywords (if present in filename)
        if "gear" in filename.lower(): caption += " due to gear damage"
        elif "volt" in filename.lower() and "over" in filename.lower(): caption += " due to over voltage"
    else:
        caption += " operating normally"

    # --- D. SECTION-SPECIFIC LOGIC (SNR per MIMII-DG specifications) ---

    # Section 00: Voltage Variation -> Fixed SNR -6.0 dB
    if section == '00':
        caption += f" at an operating voltage of {voltage}V"
        caption += ". The sound contains factory noise at -6.0 dB SNR."

    # Section 01: Weight Variation -> Fixed SNR -12.0 dB
    elif section == '01':
        caption += f" with a load weight of {weight}g"
        caption += ". The sound contains factory noise at -12.0 dB SNR."

    # Section 02: Machine ID Variation -> Fixed SNR -12.0 dB
    elif section == '02':
        caption += f" utilizing physical machine ID {machine_id}"
        caption += ". The sound contains factory noise at -12.0 dB SNR."

    return caption

# 3. MAIN PROCESSING LOOP (Iterates through splits)
for sub_split in SUB_SPLITS:

    # Input Path: .../splits/test/test_train/gearbox
    input_dir = os.path.join(TEST_SPLIT_ROOT, sub_split, MACHINE)

    # Output Path: .../text_captions/gearbox/test/test_train
    output_dir = os.path.join(CAPTION_ROOT, sub_split)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n📂 Processing Split: {sub_split.upper()}")
    print(f"   Input:  {input_dir}")
    print(f"   Output: {output_dir}")

    if not os.path.exists(input_dir):
        print(f"   ⚠️ Warning: Directory not found: {input_dir}")
        continue

    # Find all .wav files
    all_wavs = glob.glob(os.path.join(input_dir, "*.wav"))

    if not all_wavs:
        print("   ⚠️ No WAV files found.")
        continue

    print(f"   Found {len(all_wavs)} audio files.")

    total_files_split = 0

    # Process by Section
    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '') # JSON Key

            # Generate semantic text
            cap = generate_test_caption(filename, section)
            captions[key] = cap

            # Track examples for verification (Only from train split to keep console clean)
            if sub_split == 'test_train':
                if example_source is None and "source" in filename:
                    example_source = f"[{sub_split}] SOURCE: {filename}\n   TEXT: {cap}"
                if example_target is None and "target" in filename:
                    example_target = f"[{sub_split}] TARGET: {filename}\n   TEXT: {cap}"

        # Save JSON (Filename includes split name)
        json_filename = f"captions_{MACHINE}_section_{section}_{sub_split}.json"
        json_path = os.path.join(output_dir, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"   ✅ Saved: {json_filename} ({len(captions)} entries)")
        total_files_split += len(captions)

print("\n" + "="*40)
print("FINISHED GEARBOX CAPTION GENERATION.")

# --- CONSOLE OUTPUT EXAMPLES ---
print("--- GENERATED EXAMPLES ---")
if example_source:
    print(example_source)
    print("-" * 20)
if example_target:
    print(example_target)
else:
    print("WARNING: No Target domain example found.")

--- Creating Captions: GEARBOX - TEST SPLITS (TRAIN/VAL) ---

📂 Processing Split: TEST_TRAIN
   Input:  /content/drive/MyDrive/MasterProject/data/splits/test/test_train/gearbox
   Output: /content/drive/MyDrive/MasterProject/data/text_captions/gearbox/test/test_train
   Found 540 audio files.
   ✅ Saved: captions_gearbox_section_00_test_train.json (180 entries)
   ✅ Saved: captions_gearbox_section_01_test_train.json (180 entries)
   ✅ Saved: captions_gearbox_section_02_test_train.json (180 entries)

📂 Processing Split: TEST_VAL
   Input:  /content/drive/MyDrive/MasterProject/data/splits/test/test_val/gearbox
   Output: /content/drive/MyDrive/MasterProject/data/text_captions/gearbox/test/test_val
   Found 60 audio files.
   ✅ Saved: captions_gearbox_section_00_test_val.json (20 entries)
   ✅ Saved: captions_gearbox_section_01_test_val.json (20 entries)
   ✅ Saved: captions_gearbox_section_02_test_val.json (20 entries)

FINISHED GEARBOX CAPTION GENERATION.
--- GENERATED EXAMPLES ---
[tes

In [ ]:
# ============================================================
# SCRIPT: Generate Captions for SLIDER - TRAIN & VAL
# Logic: Parsing Slider-specific metadata (Velocity, Acceleration, Noise Type)
# ============================================================

print("--- Creating Captions: SLIDER - TRAIN & VAL (DEVELOPMENT) ---")

# --- Configuration ---
BASE_DIR = '/content/drive/MyDrive/MasterProject'
SPLIT_DIR = os.path.join(BASE_DIR, 'data/splits')

MACHINE = 'slider'
SECTIONS = ['00', '01', '02']
SPLITS = ['train', 'val']

# --- Prepare Directory Structure ---
# Target: data/text_captions/slider/development
OUTPUT_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE)
OUTPUT_SUBFOLDER = 'development'
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, OUTPUT_SUBFOLDER)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Storage Location: {OUTPUT_DIR}")

# Buffers for section-specific examples
examples_per_section = {
    '00': {'source': None, 'target': None},
    '01': {'source': None, 'target': None},
    '02': {'source': None, 'target': None}
}

# --- Caption Generator Function ---
def generate_caption_from_filename(filename, section):
    """
    Parses Slider metadata: Velocity (mm/s), Acceleration (m/s²), and Factory Noise.
    Constructs a semantic caption describing the physical state and noise conditions.
    """
    # --- A. PARAMETER EXTRACTION (REGEX) ---

    # 1. Slide Velocity (Section 00 focus) - Unit: mm/s
    velocity = "0"
    vel_match = re.search(r'vel_(\d+)', filename, re.IGNORECASE)
    if vel_match: velocity = vel_match.group(1)

    # 2. Acceleration (Section 01 focus) - Unit: m/s² (e.g., 0.03, 0.11)
    acceleration = "0.00"
    acc_match = re.search(r'ac[c]?_([0-9.]+)', filename, re.IGNORECASE)
    if acc_match: acceleration = acc_match.group(1)

    # 3. Factory Noise Type (Section 02 focus)
    f_noise = "A"
    fn_match = re.search(r'f-n_([A-Z])', filename, re.IGNORECASE)
    if fn_match: f_noise = fn_match.group(1)

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = "A slider"

    if is_anomaly:
        caption += " operating abnormally"
        # Optional: Specific damage detection based on keywords
        if "crack" in filename.lower(): caption += " due to cracks"
        elif "grease" in filename.lower(): caption += " due to grease removal"
        elif "belt" in filename.lower(): caption += " due to a loose belt"
    else:
        caption += " operating normally"

    # --- D. SECTION-SPECIFIC LOGIC (MIMII-DG Slider Specs) ---

    # Section 00: Velocity Variation -> Fixed -6.0 dB SNR
    if section == '00':
        caption += f" at a slide velocity of {velocity} mm/s"
        caption += ". The sound contains factory noise at -6.0 dB SNR."

    # Section 01: Acceleration Variation -> Fixed -3.0 dB SNR (Per Specifications)
    elif section == '01':
        caption += f" operating with an acceleration of {acceleration} m/s²"
        caption += ". The sound contains factory noise at -3.0 dB SNR."

    # Section 02: Factory Noise Type Variation -> Fixed -12.0 dB SNR
    elif section == '02':
        caption += f" with background factory noise type {f_noise}"
        caption += ". The sound contains factory noise at -12.0 dB SNR."

    return caption

# --- Main Processing Loop ---
total_files = 0

print(f"\nProcessing {MACHINE}...")

for split in SPLITS:
    # Resolve directory paths
    path_with_machine = os.path.join(SPLIT_DIR, split, MACHINE)
    path_direct = os.path.join(SPLIT_DIR, split)

    wav_dir = ""
    if os.path.exists(path_with_machine):
        wav_dir = path_with_machine
    elif os.path.exists(path_direct):
         if glob.glob(os.path.join(path_direct, "*.wav")):
             wav_dir = path_direct

    if not wav_dir:
        print(f"   Warning: Directory not found: {split}")
        continue

    # Gather all WAV files
    all_wavs = glob.glob(os.path.join(wav_dir, "*.wav"))
    if not all_wavs: continue

    print(f"   Split '{split}': {len(all_wavs)} files found.")

    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '')

            # Generate semantic text
            cap = generate_caption_from_filename(filename, section)
            captions[key] = cap

            # Track examples for logs
            if "source" in filename and examples_per_section[section]['source'] is None:
                examples_per_section[section]['source'] = f"FILE: {filename}\n      TEXT: {cap}"
            if "target" in filename and examples_per_section[section]['target'] is None:
                examples_per_section[section]['target'] = f"FILE: {filename}\n      TEXT: {cap}"

        # Persist results to JSON
        json_filename = f"captions_{MACHINE}_section_{section}_{split}.json"
        json_path = os.path.join(OUTPUT_DIR, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"     Saved: {json_filename} ({len(captions)} entries)")
        total_files += len(captions)

print("\n" + "="*40)
print(f"FINISHED: Total of {total_files} Slider captions (Train/Val) created.\n")

# --- Console Summary Output ---
print("--- GENERATED EXAMPLES PER SECTION ---")
for sec in SECTIONS:
    print(f"\nSECTION {sec}:")
    ex_src = examples_per_section[sec]['source']
    print(f"   [Source]: {ex_src if ex_src else 'No files found'}")
    ex_tgt = examples_per_section[sec]['target']
    print(f"   [Target]: {ex_tgt if ex_tgt else 'No files found (normal for dev set)'}")

--- Creating Captions: SLIDER - TRAIN & VAL (DEVELOPMENT) ---
Storage Location: /content/drive/MyDrive/MasterProject/data/text_captions/slider/development

Processing slider...
   Split 'train': 2727 files found.
     Saved: captions_slider_section_00_train.json (909 entries)
     Saved: captions_slider_section_01_train.json (909 entries)
     Saved: captions_slider_section_02_train.json (909 entries)
   Split 'val': 273 files found.
     Saved: captions_slider_section_00_val.json (91 entries)
     Saved: captions_slider_section_01_val.json (91 entries)
     Saved: captions_slider_section_02_val.json (91 entries)

FINISHED: Total of 3000 Slider captions (Train/Val) created.

--- GENERATED EXAMPLES PER SECTION ---

SECTION 00:
   [Source]: FILE: section_00_source_train_normal_0783_vel_300.wav
      TEXT: A slider operating normally at a slide velocity of 300 mm/s. The sound contains factory noise at -6.0 dB SNR.
   [Target]: FILE: section_00_target_train_normal_0002_vel_400.wav
      TE

In [ ]:
# ============================================================
# SCRIPT: Generate Captions for SLIDER - ADDITIONAL (03-05)
# Logic: Processing Target Domain partitions for Slider
# ============================================================

print("--- Creating Captions: SLIDER - ADDITIONAL (03-05) ---")

# --- Configuration ---
BASE_DIR = '/content/drive/MyDrive/MasterProject'
SPLIT_DIR = os.path.join(BASE_DIR, 'data/splits')

MACHINE = 'slider'
# Sections 03-05 represent the additional evaluation/target domain shifts
SECTIONS = ['03', '04', '05']
SPLITS = ['train_additional', 'val_additional']

# --- Prepare Directory Structure ---
# Target: data/text_captions/slider/additional
OUTPUT_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE)
OUTPUT_SUBFOLDER = 'additional'
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, OUTPUT_SUBFOLDER)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Storage Location: {OUTPUT_DIR}")

# Buffers for section-specific examples
examples_per_section = {
    '03': {'source': None, 'target': None},
    '04': {'source': None, 'target': None},
    '05': {'source': None, 'target': None}
}

# --- Caption Generator Function ---
def generate_caption_from_filename(filename, section):
    """
    Parses Slider additional metadata: Velocity, Acceleration, and Noise.
    Constructs a semantic caption describing the physical state and noise conditions.
    """
    # --- PARAMETER EXTRACTION (REGEX) ---

    # Velocity extraction (mm/s)
    velocity = "0"
    vel_match = re.search(r'vel_(\d+)', filename, re.IGNORECASE)
    if vel_match: velocity = vel_match.group(1)

    # Acceleration extraction (m/s²)
    acceleration = "0.00"
    acc_match = re.search(r'ac[c]?_([0-9.]+)', filename, re.IGNORECASE)
    if acc_match: acceleration = acc_match.group(1)

    # Factory Noise extraction
    f_noise = "A"
    fn_match = re.search(r'f-n_([A-Z])', filename, re.IGNORECASE)
    if fn_match: f_noise = fn_match.group(1)

    # --- STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- TEXT CONSTRUCTION ---
    caption = "A slider"

    if is_anomaly:
        caption += " operating abnormally"
        # Specific damage type detection
        if "crack" in filename.lower(): caption += " due to cracks"
        elif "grease" in filename.lower(): caption += " due to grease removal"
        elif "belt" in filename.lower(): caption += " due to a loose belt"
    else:
        caption += " operating normally"

    # --- SECTION-SPECIFIC LOGIC (MAPPING TARGET DOMAINS) ---

    # Section 03: Velocity Target (Matches Logic of Section 00)
    # SNR: -6.0 dB
    if section == '03':
        caption += f" at a slide velocity of {velocity} mm/s"
        caption += ". The sound contains factory noise at -6.0 dB SNR."

    # Section 04: Acceleration Target (Matches Logic of Section 01)
    # SNR: -3.0 dB
    elif section == '04':
        caption += f" operating with an acceleration of {acceleration} m/s²"
        caption += ". The sound contains factory noise at -3.0 dB SNR."

    # Section 05: Factory Noise Target (Matches Logic of Section 02)
    # SNR: -12.0 dB
    elif section == '05':
        caption += f" with background factory noise type {f_noise}"
        caption += ". The sound contains factory noise at -12.0 dB SNR."

    return caption

# --- Main Processing Loop ---
total_files = 0

print(f"\nProcessing {MACHINE}...")

for split_folder in SPLITS:
    path_with_machine = os.path.join(SPLIT_DIR, split_folder, MACHINE)
    path_direct = os.path.join(SPLIT_DIR, split_folder)

    wav_dir = ""
    if os.path.exists(path_with_machine):
        wav_dir = path_with_machine
    elif os.path.exists(path_direct):
         if glob.glob(os.path.join(path_direct, "*.wav")):
             wav_dir = path_direct

    if not wav_dir:
        print(f"   Warning: Directory not found: {split_folder}")
        continue

    all_wavs = glob.glob(os.path.join(wav_dir, "*.wav"))
    print(f"   Directory '{split_folder}': {len(all_wavs)} files found.")

    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '')

            # Generate semantic text
            cap = generate_caption_from_filename(filename, section)
            captions[key] = cap

            # Collect examples for logs
            if "source" in filename and examples_per_section[section]['source'] is None:
                examples_per_section[section]['source'] = f"FILE: {filename}\n      TEXT: {cap}"
            if "target" in filename and examples_per_section[section]['target'] is None:
                examples_per_section[section]['target'] = f"FILE: {filename}\n      TEXT: {cap}"

        # Persist results to JSON
        json_filename = f"captions_{MACHINE}_section_{section}_{split_folder}.json"
        json_path = os.path.join(OUTPUT_DIR, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"     Saved: {json_filename} ({len(captions)} entries)")
        total_files += len(captions)

print("\n" + "="*40)
print(f"FINISHED: Total of {total_files} Slider captions (Additional) created.\n")

# --- Console Summary Output ---
print("--- GENERATED EXAMPLES ---")
for sec in SECTIONS:
    print(f"\nSECTION {sec}:")
    ex_src = examples_per_section[sec]['source']
    print(f"   [Source]: {ex_src if ex_src else 'No files found'}")
    ex_tgt = examples_per_section[sec]['target']
    print(f"   [Target]: {ex_tgt if ex_tgt else 'No files found'}")

--- Creating Captions: SLIDER - ADDITIONAL (03-05) ---
Storage Location: /content/drive/MyDrive/MasterProject/data/text_captions/slider/additional

Processing slider...
   Directory 'train_additional': 2730 files found.
     Saved: captions_slider_section_03_train_additional.json (910 entries)
     Saved: captions_slider_section_04_train_additional.json (910 entries)
     Saved: captions_slider_section_05_train_additional.json (910 entries)
   Directory 'val_additional': 270 files found.
     Saved: captions_slider_section_03_val_additional.json (90 entries)
     Saved: captions_slider_section_04_val_additional.json (90 entries)
     Saved: captions_slider_section_05_val_additional.json (90 entries)

FINISHED: Total of 3000 Slider captions (Additional) created.

--- GENERATED EXAMPLES ---

SECTION 03:
   [Source]: FILE: section_03_source_train_normal_0037_vel_400.wav
      TEXT: A slider operating normally at a slide velocity of 400 mm/s. The sound contains factory noise at -6.0 dB SNR

In [ ]:
import os
import re
import glob
import json

# ============================================================
# SCRIPT: Generate Captions for SLIDER - TEST DATA (SPLIT)
# Logic: Processing 'test_train' and 'test_val' folders separately
#        Extracts Velocity, Acceleration, and Factory Noise
# ============================================================

print("--- Creating Captions: SLIDER - TEST SPLITS (TRAIN/VAL) ---")

# 1. Configuration
BASE_DIR = '/content/drive/MyDrive/MasterProject'

# Root of the split audio files
TEST_SPLIT_ROOT = os.path.join(BASE_DIR, 'data/splits/test')

# The two sub-splits to process
SUB_SPLITS = ['test_train', 'test_val']
MACHINE = 'slider'
SECTIONS = ['00', '01', '02']

# Target Base Directory for Captions
# Structure will be: data/text_captions/slider/test/test_train/ etc.
CAPTION_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE, 'test')

# Global variables for console examples
example_source = None
example_target = None

# 2. Caption Generator Function (Slider-Specific)
def generate_test_caption(filename, section):
    """
    Parses Slider test metadata: Velocity, Acceleration, and Factory Noise.
    Constructs a semantic caption describing the physical state and noise conditions.
    """
    # --- PARAMETER EXTRACTION (REGEX) ---
    velocity = "0"
    vel_match = re.search(r'vel_(\d+)', filename, re.IGNORECASE)
    if vel_match: velocity = vel_match.group(1)

    acceleration = "0.00"
    acc_match = re.search(r'ac[c]?_([0-9.]+)', filename, re.IGNORECASE)
    if acc_match: acceleration = acc_match.group(1)

    f_noise = "A"
    fn_match = re.search(r'f-n_([A-Z])', filename, re.IGNORECASE)
    if fn_match: f_noise = fn_match.group(1)

    # --- STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- TEXT CONSTRUCTION ---
    caption = "A slider"

    if is_anomaly:
        caption += " operating abnormally"
        # Specific fault detection (if present in filename)
        if "crack" in filename.lower(): caption += " due to cracks"
        elif "grease" in filename.lower(): caption += " due to grease removal"
        elif "belt" in filename.lower(): caption += " due to a loose belt"
    else:
        caption += " operating normally"

    # --- SECTION-SPECIFIC LOGIC ---

    # Section 00 -> SNR -6.0 dB
    if section == '00':
        caption += f" at a slide velocity of {velocity} mm/s"
        caption += ". The sound contains factory noise at -6.0 dB SNR."

    # Section 01 -> SNR -3.0 dB
    elif section == '01':
        caption += f" operating with an acceleration of {acceleration} m/s²"
        caption += ". The sound contains factory noise at -3.0 dB SNR."

    # Section 02 -> SNR -12.0 dB
    elif section == '02':
        caption += f" with background factory noise type {f_noise}"
        caption += ". The sound contains factory noise at -12.0 dB SNR."

    return caption

# 3. MAIN PROCESSING LOOP (Iterates through splits)
for sub_split in SUB_SPLITS:

    # Input Path: .../splits/test/test_train/slider
    input_dir = os.path.join(TEST_SPLIT_ROOT, sub_split, MACHINE)

    # Output Path: .../text_captions/slider/test/test_train
    output_dir = os.path.join(CAPTION_ROOT, sub_split)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n📂 Processing Split: {sub_split.upper()}")
    print(f"   Input:  {input_dir}")
    print(f"   Output: {output_dir}")

    if not os.path.exists(input_dir):
        print(f"   ⚠️ Warning: Directory not found: {input_dir}")
        continue

    # Find all .wav files
    all_wavs = glob.glob(os.path.join(input_dir, "*.wav"))

    if not all_wavs:
        print("   ⚠️ No WAV files found.")
        continue

    print(f"   Found {len(all_wavs)} audio files.")

    total_files_split = 0

    # Process by Section
    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '') # JSON Key

            # Generate semantic text
            cap = generate_test_caption(filename, section)
            captions[key] = cap

            # Track examples for verification (Only from train split)
            if sub_split == 'test_train':
                if example_source is None and "source" in filename:
                    example_source = f"[{sub_split}] SOURCE: {filename}\n   TEXT: {cap}"
                if example_target is None and "target" in filename:
                    example_target = f"[{sub_split}] TARGET: {filename}\n   TEXT: {cap}"

        # Save JSON (Filename includes split name)
        json_filename = f"captions_{MACHINE}_section_{section}_{sub_split}.json"
        json_path = os.path.join(output_dir, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"   ✅ Saved: {json_filename} ({len(captions)} entries)")
        total_files_split += len(captions)

print("\n" + "="*40)
print("FINISHED SLIDER CAPTION GENERATION.")

# --- CONSOLE OUTPUT EXAMPLES ---
print("--- GENERATED EXAMPLES ---")
if example_source:
    print(example_source)
    print("-" * 20)
if example_target:
    print(example_target)
else:
    print("WARNING: No Target domain example found.")

--- Creating Captions: SLIDER - TEST SPLITS (TRAIN/VAL) ---

📂 Processing Split: TEST_TRAIN
   Input:  /content/drive/MyDrive/MasterProject/data/splits/test/test_train/slider
   Output: /content/drive/MyDrive/MasterProject/data/text_captions/slider/test/test_train
   Found 540 audio files.
   ✅ Saved: captions_slider_section_00_test_train.json (180 entries)
   ✅ Saved: captions_slider_section_01_test_train.json (180 entries)
   ✅ Saved: captions_slider_section_02_test_train.json (180 entries)

📂 Processing Split: TEST_VAL
   Input:  /content/drive/MyDrive/MasterProject/data/splits/test/test_val/slider
   Output: /content/drive/MyDrive/MasterProject/data/text_captions/slider/test/test_val
   Found 60 audio files.
   ✅ Saved: captions_slider_section_00_test_val.json (20 entries)
   ✅ Saved: captions_slider_section_01_test_val.json (20 entries)
   ✅ Saved: captions_slider_section_02_test_val.json (20 entries)

FINISHED SLIDER CAPTION GENERATION.
--- GENERATED EXAMPLES ---
[test_train] SOU

In [ ]:
# ============================================================
# SCRIPT: Generate Captions for VALVE - TRAIN & VAL
# Logic: Parsing Valve-specific metadata (Open-Close Patterns, Panel Config)
# ============================================================

print("--- Creating Captions: VALVE - TRAIN & VAL (DEVELOPMENT) ---")

# --- Configuration ---
BASE_DIR = '/content/drive/MyDrive/MasterProject'
SPLIT_DIR = os.path.join(BASE_DIR, 'data/splits')

MACHINE = 'valve'
SECTIONS = ['00', '01', '02']
SPLITS = ['train', 'val']

# --- Prepare Directory Structure ---
# Target: data/text_captions/valve/development
OUTPUT_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE)
OUTPUT_SUBFOLDER = 'development'
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, OUTPUT_SUBFOLDER)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Storage Location: {OUTPUT_DIR}")

# Buffers for section-specific examples
examples_per_section = {
    '00': {'source': None, 'target': None},
    '01': {'source': None, 'target': None},
    '02': {'source': None, 'target': None}
}

# --- Caption Generator Function ---
def generate_caption_from_filename(filename, section):
    """
    Parses Valve metadata: Open-close patterns, panel configurations, and multi-valve setups.
    Constructs a semantic caption describing the physical state and noise conditions.
    """
    # --- A. PARAMETER EXTRACTION (REGEX) ---

    # 1. Pattern (pat) - Section 00 focus
    pattern = "00"
    pat_match = re.search(r'pat_(\d+)', filename, re.IGNORECASE)
    if pat_match: pattern = pat_match.group(1)

    # 2. Panel Configuration - Section 01 focus
    # Possible values: open, bs-c (back-side closed), b-c (back closed), s-c (side closed)
    panel = "open" # Default
    panel_match = re.search(r'_(open|bs-c|b-c|s-c)', filename, re.IGNORECASE)
    if panel_match: panel = panel_match.group(1)

    # 3. Multi-Valve Patterns (v1, v2) - Section 02 focus
    v1_pat = None
    v2_pat = None

    v1_match = re.search(r'v1_(\d+)', filename, re.IGNORECASE)
    if v1_match: v1_pat = v1_match.group(1)

    v2_match = re.search(r'v2_(\d+)', filename, re.IGNORECASE)
    if v2_match: v2_pat = v2_match.group(1)

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = "A valve"

    if is_anomaly:
        caption += " operating abnormally"
        # Specific damage detection
        if "contam" in filename.lower(): caption += " due to contamination"
        elif "stuck" in filename.lower(): caption += " due to a stuck object"
    else:
        caption += " operating normally"

    # --- D. SECTION-SPECIFIC LOGIC (SNR is constant 0.0 dB for Valve) ---

    # Section 00: Open-Close Pattern Variation
    if section == '00':
        caption += f" with open-close pattern index {pattern}"
        caption += ". The sound contains factory noise at 0.0 dB SNR."

    # Section 01: Panel Configuration Variation
    elif section == '01':
        # Translate technical abbreviations for readability
        panel_desc = panel
        if panel == "bs-c": panel_desc = "back-side closed"
        elif panel == "b-c": panel_desc = "back closed"
        elif panel == "s-c": panel_desc = "side closed"
        elif panel == "open": panel_desc = "open (no panels)"

        caption += f" with panel configuration '{panel_desc}'"
        caption += ". The sound contains factory noise at 0.0 dB SNR."

    # Section 02: Multi-Valve Interaction Variation
    elif section == '02':
        valve_parts = []
        if v1_pat: valve_parts.append(f"valve 1 pattern {v1_pat}")
        if v2_pat: valve_parts.append(f"valve 2 pattern {v2_pat}")

        if valve_parts:
            caption += f" using {', and '.join(valve_parts)}"
        else:
            caption += " using a specific valve pattern"

        caption += ". The sound contains factory noise at 0.0 dB SNR."

    return caption

# --- Main Processing Loop ---
total_files = 0

print(f"\nProcessing {MACHINE}...")

for split in SPLITS:
    # Resolve directory paths
    path_with_machine = os.path.join(SPLIT_DIR, split, MACHINE)
    path_direct = os.path.join(SPLIT_DIR, split)

    wav_dir = ""
    if os.path.exists(path_with_machine):
        wav_dir = path_with_machine
    elif os.path.exists(path_direct):
         if glob.glob(os.path.join(path_direct, "*.wav")):
             wav_dir = path_direct

    if not wav_dir:
        print(f"   Warning: Directory not found: {split}")
        continue

    all_wavs = glob.glob(os.path.join(wav_dir, "*.wav"))
    if not all_wavs: continue

    print(f"   Split '{split}': {len(all_wavs)} files found.")

    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '')

            # Generate semantic text
            cap = generate_caption_from_filename(filename, section)
            captions[key] = cap

            # Collect examples for logs
            if "source" in filename and examples_per_section[section]['source'] is None:
                examples_per_section[section]['source'] = f"FILE: {filename}\n      TEXT: {cap}"
            if "target" in filename and examples_per_section[section]['target'] is None:
                examples_per_section[section]['target'] = f"FILE: {filename}\n      TEXT: {cap}"

        # Persist results to JSON
        json_filename = f"captions_{MACHINE}_section_{section}_{split}.json"
        json_path = os.path.join(OUTPUT_DIR, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"     Saved: {json_filename} ({len(captions)} entries)")
        total_files += len(captions)

print("\n" + "="*40)
print(f"FINISHED: Total of {total_files} Valve captions (Train/Val) created.\n")

# --- Console Summary Output ---
print("--- GENERATED EXAMPLES PER SECTION ---")
for sec in SECTIONS:
    print(f"\nSECTION {sec}:")

    ex_src = examples_per_section[sec]['source']
    if ex_src:
        print(f"   [Source]:\n      {ex_src}")
    else:
        print("   [Source]: No files found.")

    ex_tgt = examples_per_section[sec]['target']
    if ex_tgt:
        print(f"   [Target]:\n      {ex_tgt}")
    else:
        print("   [Target]: No files found (normal for development set).")

--- Creating Captions: VALVE - TRAIN & VAL (DEVELOPMENT) ---
Storage Location: /content/drive/MyDrive/MasterProject/data/text_captions/valve/development

Processing valve...
   Split 'train': 2727 files found.
     Saved: captions_valve_section_00_train.json (909 entries)
     Saved: captions_valve_section_01_train.json (909 entries)
     Saved: captions_valve_section_02_train.json (909 entries)
   Split 'val': 273 files found.
     Saved: captions_valve_section_00_val.json (91 entries)
     Saved: captions_valve_section_01_val.json (91 entries)
     Saved: captions_valve_section_02_val.json (91 entries)

FINISHED: Total of 3000 Valve captions (Train/Val) created.

--- GENERATED EXAMPLES PER SECTION ---

SECTION 00:
   [Source]:
      FILE: section_00_source_train_normal_0502_pat_00.wav
      TEXT: A valve operating normally with open-close pattern index 00. The sound contains factory noise at 0.0 dB SNR.
   [Target]:
      FILE: section_00_target_train_normal_0001_pat_02.wav
      TEX

In [ ]:
# ============================================================
# SCRIPT: Generate Captions for VALVE - ADDITIONAL (03-05)
# Logic: Processing Target Domain partitions for Valve
# ============================================================

print("--- Creating Captions: VALVE - ADDITIONAL (03-05) WITH EXAMPLES ---")

# --- Configuration ---
BASE_DIR = '/content/drive/MyDrive/MasterProject'
SPLIT_DIR = os.path.join(BASE_DIR, 'data/splits')

MACHINE = 'valve'
# Sections 03-05 represent target domain shifts for generalization
SECTIONS = ['03', '04', '05']
SPLITS = ['train_additional', 'val_additional']

# --- Prepare Directory Structure ---
# Target: data/text_captions/valve/additional
OUTPUT_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE)
OUTPUT_SUBFOLDER = 'additional'
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, OUTPUT_SUBFOLDER)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Storage Location: {OUTPUT_DIR}")

# Buffers for section-specific examples
examples_per_section = {
    '03': {'source': None, 'target': None},
    '04': {'source': None, 'target': None},
    '05': {'source': None, 'target': None}
}

# --- Caption Generator Function ---
def generate_caption_from_filename(filename, section):
    """
    Parses Valve additional metadata: Patterns, panel configurations, and multi-valve setups.
    Describes the physical state of the target domain shifts.
    """
    # --- A. PARAMETER EXTRACTION (REGEX) ---

    # 1. Pattern (Section 03 targets - Mapping to logic of Section 00)
    pattern = "00"
    pat_match = re.search(r'pat_(\d+)', filename, re.IGNORECASE)
    if pat_match: pattern = pat_match.group(1)

    # 2. Panel Configuration (Section 04 targets - Mapping to logic of Section 01)
    panel = "open"
    panel_match = re.search(r'_(open|bs-c|b-c|s-c)', filename, re.IGNORECASE)
    if panel_match: panel = panel_match.group(1)

    # 3. Multi-Valve setup (Section 05 targets - Mapping to logic of Section 02)
    v1_pat = None
    v2_pat = None
    v1_match = re.search(r'v1_(\d+)', filename, re.IGNORECASE)
    if v1_match: v1_pat = v1_match.group(1)
    v2_match = re.search(r'v2_(\d+)', filename, re.IGNORECASE)
    if v2_match: v2_pat = v2_match.group(1)

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = "A valve"

    if is_anomaly:
        caption += " operating abnormally"
        if "contam" in filename.lower(): caption += " due to contamination"
        elif "stuck" in filename.lower(): caption += " due to a stuck object"
    else:
        caption += " operating normally"

    # --- D. SECTION-SPECIFIC LOGIC (TARGET MAPPING) ---
    # Descriptions reflect physical state. SNR for Valve is consistently 0.0 dB.

    # Section 03 -> Target Pattern Variation
    if section == '03':
        caption += f" with open-close pattern index {pattern}"
        caption += ". The sound contains factory noise at 0.0 dB SNR."

    # Section 04 -> Target Panel Configuration
    elif section == '04':
        panel_desc = panel
        if panel == "bs-c": panel_desc = "back-side closed"
        elif panel == "b-c": panel_desc = "back closed"
        elif panel == "s-c": panel_desc = "side closed"

        caption += f" with panel configuration '{panel_desc}'"
        caption += ". The sound contains factory noise at 0.0 dB SNR."

    # Section 05 -> Target Multi-Valve setup
    elif section == '05':
        valve_parts = []
        if v1_pat: valve_parts.append(f"valve 1 pattern {v1_pat}")
        if v2_pat: valve_parts.append(f"valve 2 pattern {v2_pat}")

        if valve_parts:
            caption += f" using {', and '.join(valve_parts)}"
        else:
            caption += " using a specific valve pattern"

        caption += ". The sound contains factory noise at 0.0 dB SNR."

    return caption

# --- Main Processing Loop ---
total_files = 0

print(f"\nProcessing {MACHINE}...")

for split_folder in SPLITS:
    path_with_machine = os.path.join(SPLIT_DIR, split_folder, MACHINE)
    path_direct = os.path.join(SPLIT_DIR, split_folder)

    wav_dir = ""
    if os.path.exists(path_with_machine):
        wav_dir = path_with_machine
    elif os.path.exists(path_direct):
         if glob.glob(os.path.join(path_direct, "*.wav")):
             wav_dir = path_direct

    if not wav_dir:
        print(f"   Warning: Directory not found: {split_folder}")
        continue

    all_wavs = glob.glob(os.path.join(wav_dir, "*.wav"))
    if not all_wavs: continue

    print(f"   Directory '{split_folder}': {len(all_wavs)} files found.")

    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '')

            # Generate semantic text
            cap = generate_caption_from_filename(filename, section)
            captions[key] = cap

            # Track examples for logs
            if "source" in filename and examples_per_section[section]['source'] is None:
                examples_per_section[section]['source'] = f"FILE: {filename}\n      TEXT: {cap}"
            if "target" in filename and examples_per_section[section]['target'] is None:
                examples_per_section[section]['target'] = f"FILE: {filename}\n      TEXT: {cap}"

        # Persist results to JSON
        json_filename = f"captions_{MACHINE}_section_{section}_{split_folder}.json"
        json_path = os.path.join(OUTPUT_DIR, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"     Saved: {json_filename} ({len(captions)} entries)")
        total_files += len(captions)

print("\n" + "="*40)
print(f"FINISHED: Total of {total_files} Valve captions (Additional) created.\n")

print("--- GENERATED EXAMPLES PER SECTION ---")
for sec in SECTIONS:
    print(f"\nSECTION {sec}:")
    ex_src = examples_per_section[sec]['source']
    print(f"   [Source]: {ex_src if ex_src else 'No files found'}")
    ex_tgt = examples_per_section[sec]['target']
    print(f"   [Target]: {ex_tgt if ex_tgt else 'No files found'}")

--- Creating Captions: VALVE - ADDITIONAL (03-05) WITH EXAMPLES ---
Storage Location: /content/drive/MyDrive/MasterProject/data/text_captions/valve/additional

Processing valve...
   Directory 'train_additional': 2730 files found.
     Saved: captions_valve_section_03_train_additional.json (910 entries)
     Saved: captions_valve_section_04_train_additional.json (910 entries)
     Saved: captions_valve_section_05_train_additional.json (910 entries)
   Directory 'val_additional': 270 files found.
     Saved: captions_valve_section_03_val_additional.json (90 entries)
     Saved: captions_valve_section_04_val_additional.json (90 entries)
     Saved: captions_valve_section_05_val_additional.json (90 entries)

FINISHED: Total of 3000 Valve captions (Additional) created.

--- GENERATED EXAMPLES PER SECTION ---

SECTION 03:
   [Source]: FILE: section_03_source_train_normal_0710_pat_00.wav
      TEXT: A valve operating normally with open-close pattern index 00. The sound contains factory noise

In [ ]:
import os
import re
import glob
import json

# ============================================================
# SCRIPT: Generate Captions for VALVE - TEST DATA (SPLIT)
# Logic: Processing 'test_train' and 'test_val' folders separately
#        Extracts Patterns, Panel Configs, and Multi-Valve logic
# ============================================================

print("--- Creating Captions: VALVE - TEST SPLITS (TRAIN/VAL) ---")

# 1. Configuration
BASE_DIR = '/content/drive/MyDrive/MasterProject'

# Root of the split audio files
TEST_SPLIT_ROOT = os.path.join(BASE_DIR, 'data/splits/test')

# The two sub-splits to process
SUB_SPLITS = ['test_train', 'test_val']
MACHINE = 'valve'
SECTIONS = ['00', '01', '02']

# Target Base Directory for Captions
# Structure will be: data/text_captions/valve/test/test_train/ etc.
CAPTION_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE, 'test')

# Global variables for console examples
example_source = None
example_target = None

# 2. Caption Generator Function (Valve-Specific)
def generate_test_caption(filename, section):
    """
    Parses Valve test metadata: Open-close patterns, panel configurations, and multi-valve setups.
    Constructs a semantic caption describing the physical state and noise conditions.
    """
    # --- A. PARAMETER EXTRACTION (REGEX) ---

    # 1. Pattern (Section 00)
    pattern = "00"
    pat_match = re.search(r'pat_(\d+)', filename, re.IGNORECASE)
    if pat_match: pattern = pat_match.group(1)

    # 2. Panel (Section 01)
    panel = "open"
    panel_match = re.search(r'_(open|bs-c|b-c|s-c)', filename, re.IGNORECASE)
    if panel_match: panel = panel_match.group(1)

    # 3. Multi-Valve (Section 02)
    v1_pat = None
    v2_pat = None
    v1_match = re.search(r'v1_(\d+)', filename, re.IGNORECASE)
    if v1_match: v1_pat = v1_match.group(1)
    v2_match = re.search(r'v2_(\d+)', filename, re.IGNORECASE)
    if v2_match: v2_pat = v2_match.group(1)

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = "A valve"

    if is_anomaly:
        caption += " operating abnormally"
        # Specific fault detection
        if "contam" in filename.lower(): caption += " due to contamination"
        elif "stuck" in filename.lower(): caption += " due to a stuck object"
    else:
        caption += " operating normally"

    # --- D. SECTION-SPECIFIC LOGIC (SNR is constant 0.0 dB for Valve) ---

    # Section 00: Pattern Variation
    if section == '00':
        caption += f" with open-close pattern index {pattern}"
        caption += ". The sound contains factory noise at 0.0 dB SNR."

    # Section 01: Panel Variation
    elif section == '01':
        # Translate technical codes for better readability
        panel_desc = panel
        if panel == "bs-c": panel_desc = "back-side closed"
        elif panel == "b-c": panel_desc = "back closed"
        elif panel == "s-c": panel_desc = "side closed"
        elif panel == "open": panel_desc = "open (no panels)"

        caption += f" with panel configuration '{panel_desc}'"
        caption += ". The sound contains factory noise at 0.0 dB SNR."

    # Section 02: Multi-Valve Variation
    elif section == '02':
        valve_parts = []
        if v1_pat: valve_parts.append(f"valve 1 pattern {v1_pat}")
        if v2_pat: valve_parts.append(f"valve 2 pattern {v2_pat}")

        if valve_parts:
            caption += f" using {', and '.join(valve_parts)}"
        else:
            caption += " using a specific valve pattern"

        caption += ". The sound contains factory noise at 0.0 dB SNR."

    return caption

# 3. MAIN PROCESSING LOOP (Iterates through splits)
for sub_split in SUB_SPLITS:

    # Input Path: .../splits/test/test_train/valve
    input_dir = os.path.join(TEST_SPLIT_ROOT, sub_split, MACHINE)

    # Output Path: .../text_captions/valve/test/test_train
    output_dir = os.path.join(CAPTION_ROOT, sub_split)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n📂 Processing Split: {sub_split.upper()}")
    print(f"   Input:  {input_dir}")
    print(f"   Output: {output_dir}")

    if not os.path.exists(input_dir):
        print(f"   ⚠️ Warning: Directory not found: {input_dir}")
        continue

    # Find all .wav files
    all_wavs = glob.glob(os.path.join(input_dir, "*.wav"))

    if not all_wavs:
        print("   ⚠️ No WAV files found.")
        continue

    print(f"   Found {len(all_wavs)} audio files.")

    total_files_split = 0

    # Process by Section
    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '') # JSON Key

            # Generate semantic text
            cap = generate_test_caption(filename, section)
            captions[key] = cap

            # Track examples for verification (Only from train split)
            if sub_split == 'test_train':
                if example_source is None and "source" in filename:
                    example_source = f"[{sub_split}] SOURCE: {filename}\n   TEXT: {cap}"
                if example_target is None and "target" in filename:
                    example_target = f"[{sub_split}] TARGET: {filename}\n   TEXT: {cap}"

        # Save JSON (Filename includes split name)
        json_filename = f"captions_{MACHINE}_section_{section}_{sub_split}.json"
        json_path = os.path.join(output_dir, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"   ✅ Saved: {json_filename} ({len(captions)} entries)")
        total_files_split += len(captions)

print("\n" + "="*40)
print("FINISHED VALVE CAPTION GENERATION.")

# --- CONSOLE OUTPUT EXAMPLES ---
print("--- GENERATED EXAMPLES ---")
if example_source:
    print(example_source)
    print("-" * 20)
if example_target:
    print(example_target)
else:
    print("WARNING: No Target domain example found.")

--- Creating Captions: VALVE - TEST SPLITS (TRAIN/VAL) ---

📂 Processing Split: TEST_TRAIN
   Input:  /content/drive/MyDrive/MasterProject/data/splits/test/test_train/valve
   Output: /content/drive/MyDrive/MasterProject/data/text_captions/valve/test/test_train
   Found 540 audio files.
   ✅ Saved: captions_valve_section_00_test_train.json (180 entries)
   ✅ Saved: captions_valve_section_01_test_train.json (180 entries)
   ✅ Saved: captions_valve_section_02_test_train.json (180 entries)

📂 Processing Split: TEST_VAL
   Input:  /content/drive/MyDrive/MasterProject/data/splits/test/test_val/valve
   Output: /content/drive/MyDrive/MasterProject/data/text_captions/valve/test/test_val
   Found 60 audio files.
   ✅ Saved: captions_valve_section_00_test_val.json (20 entries)
   ✅ Saved: captions_valve_section_01_test_val.json (20 entries)
   ✅ Saved: captions_valve_section_02_test_val.json (20 entries)

FINISHED VALVE CAPTION GENERATION.
--- GENERATED EXAMPLES ---
[test_train] SOURCE: section

In [ ]:
# ============================================================
# SCRIPT: Generate Captions for TOYCAR - TRAIN & VAL
# Logic: Parsing ToyCar-specific metadata (Car Model, Voltage, Mic ID)
# ============================================================

print("--- Creating Captions: TOYCAR - TRAIN & VAL (CORRECTED MODELS) ---")

# --- Configuration ---
BASE_DIR = '/content/drive/MyDrive/MasterProject'
SPLIT_DIR = os.path.join(BASE_DIR, 'data/splits')

MACHINE = 'ToyCar'
SECTIONS = ['00', '01', '02']
SPLITS = ['train', 'val']

# --- Prepare Directory Structure ---
# Target: data/text_captions/ToyCar/development
OUTPUT_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE)
OUTPUT_SUBFOLDER = 'development'
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, OUTPUT_SUBFOLDER)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Storage Location: {OUTPUT_DIR}")

# Buffers for section-specific examples
examples_per_section = {
    '00': {'source': None, 'target': None},
    '01': {'source': None, 'target': None},
    '02': {'source': None, 'target': None}
}

# --- Caption Generator Function ---
def generate_caption_from_filename(filename, section):
    """
    Parses ToyCar metadata: Car model (alpha-numeric), operating voltage,
    microphone ID, and noise patterns. Construct semantic descriptions.
    """
    # --- A. PARAMETER EXTRACTION (REGEX) ---

    # 1. Car Model (car) - Supports alphanumeric identifiers (e.g., A, A1, B2)
    car_model = "A"
    car_match = re.search(r'car_([A-Z0-9]+)', filename, re.IGNORECASE)
    if car_match: car_model = car_match.group(1).upper()

    # 2. Speed / Voltage (spd) - Normalized by dividing by 10.0
    voltage = "3.0"
    spd_match = re.search(r'spd_(\d+)', filename, re.IGNORECASE)
    if spd_match:
        raw_val = int(spd_match.group(1))
        voltage = str(raw_val / 10.0)

    # 3. Microphone ID (mic)
    mic_id = "1"
    mic_match = re.search(r'mic_(\d+)', filename, re.IGNORECASE)
    if mic_match: mic_id = mic_match.group(1)

    # 4. Noise Pattern (noise or n)
    noise_id = "1"
    noise_match = re.search(r'(?:noise|n)_(\d+)', filename, re.IGNORECASE)
    if noise_match: noise_id = noise_match.group(1)

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = f"A miniature 4WD ToyCar model {car_model}"

    if is_anomaly:
        caption += " operating abnormally due to bent shaft, damaged gears, or wheel damage"
    else:
        caption += " operating normally"

    # --- D. TECHNICAL DETAILS ---
    caption += f" at an operating voltage of {voltage} V"
    caption += f" recorded with microphone {mic_id}"
    caption += f" mixed with background noise pattern {noise_id}."

    return caption

# --- Main Processing Loop ---
total_files = 0

print(f"\nProcessing {MACHINE}...")

for split in SPLITS:
    # Resolve directory paths for current split
    path_with_machine = os.path.join(SPLIT_DIR, split, MACHINE)
    path_direct = os.path.join(SPLIT_DIR, split)

    wav_dir = ""
    if os.path.exists(path_with_machine):
        wav_dir = path_with_machine
    elif os.path.exists(path_direct):
         for folder in os.listdir(path_direct):
             if folder.lower() == MACHINE.lower():
                 wav_dir = os.path.join(path_direct, folder)
                 break

    if not wav_dir:
        print(f"   Warning: Directory not found for split: {split}")
        continue

    all_wavs = glob.glob(os.path.join(wav_dir, "*.wav"))
    if not all_wavs: continue

    print(f"   Split '{split}': {len(all_wavs)} files found.")

    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '')

            # Generate semantic caption
            cap = generate_caption_from_filename(filename, section)
            captions[key] = cap

            # Collect examples for log verification
            if "source" in filename and examples_per_section[section]['source'] is None:
                examples_per_section[section]['source'] = f"FILE: {filename}\n      TEXT: {cap}"
            if "target" in filename and examples_per_section[section]['target'] is None:
                examples_per_section[section]['target'] = f"FILE: {filename}\n      TEXT: {cap}"

        # Persist results to JSON
        json_filename = f"captions_{MACHINE}_section_{section}_{split}.json"
        json_path = os.path.join(OUTPUT_DIR, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"     Saved: {json_filename} ({len(captions)} entries)")
        total_files += len(captions)

print("\n" + "="*40)
print(f"FINISHED: Total of {total_files} ToyCar captions (Train/Val) created.\n")

# --- Console Summary Output ---
print("--- GENERATED EXAMPLES PER SECTION ---")
for sec in SECTIONS:
    print(f"\nSECTION {sec}:")
    ex_src = examples_per_section[sec]['source']
    print(f"   [Source]: {ex_src if ex_src else 'No files found'}")
    ex_tgt = examples_per_section[sec]['target']
    print(f"   [Target]: {ex_tgt if ex_tgt else 'No files found (normal for development split)'}")

--- Creating Captions: TOYCAR - TRAIN & VAL (CORRECTED MODELS) ---
Storage Location: /content/drive/MyDrive/MasterProject/data/text_captions/ToyCar/development

Processing ToyCar...
   Split 'train': 2727 files found.
     Saved: captions_ToyCar_section_00_train.json (909 entries)
     Saved: captions_ToyCar_section_01_train.json (909 entries)
     Saved: captions_ToyCar_section_02_train.json (909 entries)
   Split 'val': 273 files found.
     Saved: captions_ToyCar_section_00_val.json (91 entries)
     Saved: captions_ToyCar_section_01_val.json (91 entries)
     Saved: captions_ToyCar_section_02_val.json (91 entries)

FINISHED: Total of 3000 ToyCar captions (Train/Val) created.

--- GENERATED EXAMPLES PER SECTION ---

SECTION 00:
   [Source]: FILE: section_00_source_train_normal_0978_car_E2_spd_34V_mic_1_noise_1.wav
      TEXT: A miniature 4WD ToyCar model E2 operating normally at an operating voltage of 3.4 V recorded with microphone 1 mixed with background noise pattern 00.
   [Targ

In [ ]:
# ============================================================
# SCRIPT: Generate Captions for TOYCAR - ADDITIONAL (03-05)
# Logic: Processing Additional Training Data partitions for ToyCar
# ============================================================

print("--- Creating Captions: TOYCAR - ADDITIONAL (03-05) ---")

# --- Configuration ---
BASE_DIR = '/content/drive/MyDrive/MasterProject'
SPLIT_DIR = os.path.join(BASE_DIR, 'data/splits')

MACHINE = 'ToyCar'
# Sections 03-05 represent the domain-shift partitions for additional training
SECTIONS = ['03', '04', '05']
SPLITS = ['train_additional', 'val_additional']

# --- Prepare Directory Structure ---
# Target: data/text_captions/ToyCar/additional
OUTPUT_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE)
OUTPUT_SUBFOLDER = 'additional'
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, OUTPUT_SUBFOLDER)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Storage Location: {OUTPUT_DIR}")

# Buffers for section-specific examples
examples_per_section = {
    '03': {'source': None, 'target': None},
    '04': {'source': None, 'target': None},
    '05': {'source': None, 'target': None}
}

# --- Caption Generator Function ---
def generate_caption_from_filename(filename, section):
    """
    Parses ToyCar metadata from filenames to generate semantic text captions.
    Extracts car model, operating voltage, microphone ID, and noise patterns.
    """
    # --- A. PARAMETER EXTRACTION (REGEX) ---

    # 1. Car Model (car) - Supports alphanumeric (e.g., A2, B)
    car_model = "A"
    car_match = re.search(r'car_([A-Z0-9]+)', filename, re.IGNORECASE)
    if car_match: car_model = car_match.group(1).upper()

    # 2. Speed / Voltage (spd) - Normalized (divided by 10)
    voltage = "3.0"
    spd_match = re.search(r'spd_(\d+)', filename, re.IGNORECASE)
    if spd_match:
        # e.g., speed value 34 becomes 3.4V
        voltage = str(int(spd_match.group(1)) / 10.0)

    # 3. Microphone ID (mic)
    mic_id = "1"
    mic_match = re.search(r'mic_(\d+)', filename, re.IGNORECASE)
    if mic_match: mic_id = mic_match.group(1)

    # 4. Noise Pattern (noise or n)
    noise_id = "1"
    noise_match = re.search(r'(?:noise|n)_(\d+)', filename, re.IGNORECASE)
    if noise_match: noise_id = noise_match.group(1)

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = f"A miniature 4WD ToyCar model {car_model}"

    if is_anomaly:
        caption += " operating abnormally due to bent shaft, damaged gears, or wheel damage"
    else:
        caption += " operating normally"

    # --- D. TECHNICAL DETAILS ---
    caption += f" at an operating voltage of {voltage} V"
    caption += f" recorded with microphone {mic_id}"
    caption += f" mixed with background noise pattern {noise_id}."

    return caption

# --- Main Processing Loop ---
total_files = 0

print(f"\nProcessing {MACHINE}...")

for split_folder in SPLITS:
    # Resolve directory path
    path_with_machine = os.path.join(SPLIT_DIR, split_folder, MACHINE)
    path_direct = os.path.join(SPLIT_DIR, split_folder)

    wav_dir = ""
    if os.path.exists(path_with_machine):
        wav_dir = path_with_machine
    elif os.path.exists(path_direct):
         for folder in os.listdir(path_direct):
             if folder.lower() == MACHINE.lower():
                 wav_dir = os.path.join(path_direct, folder)
                 break

    if not wav_dir:
        print(f"   Warning: Directory not found for split: {split_folder}")
        continue

    all_wavs = glob.glob(os.path.join(wav_dir, "*.wav"))
    if not all_wavs: continue

    print(f"   Directory '{split_folder}': {len(all_wavs)} files found.")

    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '')

            # Generate semantic caption
            cap = generate_caption_from_filename(filename, section)
            captions[key] = cap

            # Track examples for logs
            if "source" in filename and examples_per_section[section]['source'] is None:
                examples_per_section[section]['source'] = f"FILE: {filename}\n      TEXT: {cap}"
            if "target" in filename and examples_per_section[section]['target'] is None:
                examples_per_section[section]['target'] = f"FILE: {filename}\n      TEXT: {cap}"

        # Persist results to JSON
        json_filename = f"captions_{MACHINE}_section_{section}_{split_folder}.json"
        json_path = os.path.join(OUTPUT_DIR, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"     Saved: {json_filename} ({len(captions)} entries)")
        total_files += len(captions)

print("\n" + "="*40)
print(f"FINISHED: Total of {total_files} ToyCar captions (Additional) created.\n")

# --- Console Summary Output ---
print("--- GENERATED EXAMPLES PER SECTION ---")
for sec in SECTIONS:
    print(f"\nSECTION {sec}:")
    ex_src = examples_per_section[sec]['source']
    print(f"   [Source]: {ex_src if ex_src else 'No files found'}")
    ex_tgt = examples_per_section[sec]['target']
    print(f"   [Target]: {ex_tgt if ex_tgt else 'No files found'}")

--- Creating Captions: TOYCAR - ADDITIONAL (03-05) ---
Storage Location: /content/drive/MyDrive/MasterProject/data/text_captions/ToyCar/additional

Processing ToyCar...
   Directory 'train_additional': 2730 files found.
     Saved: captions_ToyCar_section_03_train_additional.json (910 entries)
     Saved: captions_ToyCar_section_04_train_additional.json (910 entries)
     Saved: captions_ToyCar_section_05_train_additional.json (910 entries)
   Directory 'val_additional': 270 files found.
     Saved: captions_ToyCar_section_03_val_additional.json (90 entries)
     Saved: captions_ToyCar_section_04_val_additional.json (90 entries)
     Saved: captions_ToyCar_section_05_val_additional.json (90 entries)

FINISHED: Total of 3000 ToyCar captions (Additional) created.

--- GENERATED EXAMPLES PER SECTION ---

SECTION 03:
   [Source]: FILE: section_03_source_train_normal_0191_car_F2_spd_31V_mic_1_noise_6.wav
      TEXT: A miniature 4WD ToyCar model F2 operating normally at an operating voltage 

In [ ]:
import os
import re
import glob
import json

# ============================================================
# SCRIPT: Generate Captions for TOYCAR - TEST DATA (SPLIT)
# Logic: Processing 'test_train' and 'test_val' folders separately
#        Extracts Car Model, Voltage, Mic ID, and Noise Pattern
# ============================================================

print("--- Creating Captions: TOYCAR - TEST SPLITS (TRAIN/VAL) ---")

# 1. Configuration
BASE_DIR = '/content/drive/MyDrive/MasterProject'

# Root of the split audio files
TEST_SPLIT_ROOT = os.path.join(BASE_DIR, 'data/splits/test')

# The two sub-splits to process
SUB_SPLITS = ['test_train', 'test_val']
MACHINE = 'ToyCar'
SECTIONS = ['00', '01', '02']

# Target Base Directory for Captions
# Structure will be: data/text_captions/ToyCar/test/test_train/ etc.
CAPTION_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE, 'test')

# Global variables for console examples
example_source = None
example_target = None

# 2. Caption Generator Function (ToyCar-Specific)
def generate_test_caption(filename, section):
    """
    Parses ToyCar test metadata: Car model, normalized voltage,
    microphone ID, and noise patterns. Construct semantic descriptions.
    """
    # --- A. PARAMETER EXTRACTION (REGEX) ---

    # 1. Car Model (e.g., A, B, A2)
    car_model = "A"
    car_match = re.search(r'car_([A-Z0-9]+)', filename, re.IGNORECASE)
    if car_match: car_model = car_match.group(1).upper()

    # 2. Speed / Voltage - Normalized (divided by 10)
    # The filename usually contains 'spd_1' which corresponds to X voltage steps
    voltage = "3.0"
    spd_match = re.search(r'spd_(\d+)', filename, re.IGNORECASE)
    if spd_match:
        # In ToyCar dataset, 'spd' value is often mapped to voltage directly or via a factor
        # Common practice in MIMII-DG processing: value / 10.0
        voltage = str(int(spd_match.group(1)) / 10.0)

    # 3. Microphone ID
    mic_id = "1"
    mic_match = re.search(r'mic_(\d+)', filename, re.IGNORECASE)
    if mic_match: mic_id = mic_match.group(1)

    # 4. Noise Pattern ID
    noise_id = "1"
    noise_match = re.search(r'(?:noise|n)_(\d+)', filename, re.IGNORECASE)
    if noise_match: noise_id = noise_match.group(1)

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = f"A miniature 4WD ToyCar model {car_model}"

    if is_anomaly:
        caption += " operating abnormally"
    else:
        caption += " operating normally"

    # --- D. TECHNICAL SPECIFICATIONS ---
    caption += f" at an operating voltage of {voltage} V"
    caption += f" recorded with microphone {mic_id}"
    caption += f" mixed with background noise pattern {noise_id}."

    return caption

# 3. MAIN PROCESSING LOOP (Iterates through splits)
for sub_split in SUB_SPLITS:

    # Input Path: .../splits/test/test_train/ToyCar (Need to handle case sensitivity)
    # The split script likely created the folder as 'ToyCar' (mixed case) or 'toycar' (lower)
    # We check both to be safe.
    input_base = os.path.join(TEST_SPLIT_ROOT, sub_split)
    input_dir = None

    # Locate actual directory (ToyCar vs toycar)
    if os.path.exists(input_base):
        for folder in os.listdir(input_base):
            if folder.lower() == MACHINE.lower():
                input_dir = os.path.join(input_base, folder)
                break

    # Output Path: .../text_captions/ToyCar/test/test_train
    output_dir = os.path.join(CAPTION_ROOT, sub_split)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n📂 Processing Split: {sub_split.upper()}")
    print(f"   Input:  {input_dir}")
    print(f"   Output: {output_dir}")

    if not input_dir or not os.path.exists(input_dir):
        print(f"   ⚠️ Warning: Directory not found: {input_dir}")
        continue

    # Find all .wav files
    all_wavs = glob.glob(os.path.join(input_dir, "*.wav"))

    if not all_wavs:
        print("   ⚠️ No WAV files found.")
        continue

    print(f"   Found {len(all_wavs)} audio files.")

    total_files_split = 0

    # Process by Section
    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '') # JSON Key

            # Generate semantic text
            cap = generate_test_caption(filename, section)
            captions[key] = cap

            # Track examples for verification (Only from train split)
            if sub_split == 'test_train':
                if example_source is None and "source" in filename:
                    example_source = f"[{sub_split}] SOURCE: {filename}\n   TEXT: {cap}"
                if example_target is None and "target" in filename:
                    example_target = f"[{sub_split}] TARGET: {filename}\n   TEXT: {cap}"

        # Save JSON (Filename includes split name)
        json_filename = f"captions_{MACHINE}_section_{section}_{sub_split}.json"
        json_path = os.path.join(output_dir, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"   ✅ Saved: {json_filename} ({len(captions)} entries)")
        total_files_split += len(captions)

print("\n" + "="*40)
print("FINISHED TOYCAR CAPTION GENERATION.")

# --- CONSOLE OUTPUT EXAMPLES ---
print("--- GENERATED EXAMPLES ---")
if example_source:
    print(example_source)
    print("-" * 20)
if example_target:
    print(example_target)
else:
    print("WARNING: No Target domain example found.")

--- Creating Captions: TOYCAR - TEST SPLITS (TRAIN/VAL) ---

📂 Processing Split: TEST_TRAIN
   Input:  /content/drive/MyDrive/MasterProject/data/splits/test/test_train/ToyCar
   Output: /content/drive/MyDrive/MasterProject/data/text_captions/ToyCar/test/test_train
   Found 540 audio files.
   ✅ Saved: captions_ToyCar_section_00_test_train.json (180 entries)
   ✅ Saved: captions_ToyCar_section_01_test_train.json (180 entries)
   ✅ Saved: captions_ToyCar_section_02_test_train.json (180 entries)

📂 Processing Split: TEST_VAL
   Input:  /content/drive/MyDrive/MasterProject/data/splits/test/test_val/ToyCar
   Output: /content/drive/MyDrive/MasterProject/data/text_captions/ToyCar/test/test_val
   Found 60 audio files.
   ✅ Saved: captions_ToyCar_section_00_test_val.json (20 entries)
   ✅ Saved: captions_ToyCar_section_01_test_val.json (20 entries)
   ✅ Saved: captions_ToyCar_section_02_test_val.json (20 entries)

FINISHED TOYCAR CAPTION GENERATION.
--- GENERATED EXAMPLES ---
[test_train] SOU

In [ ]:
# ============================================================
# SCRIPT: Generate Captions for TOYTRAIN - TRAIN & VAL
# Logic: Parsing ToyTrain-specific metadata (Model, Speed, Mic Position)
# ============================================================

print("--- Creating Captions: TOYTRAIN - TRAIN & VAL (CORRECTED MODELS) ---")

# --- Configuration ---
BASE_DIR = '/content/drive/MyDrive/MasterProject'
SPLIT_DIR = os.path.join(BASE_DIR, 'data/splits')

MACHINE = 'ToyTrain'
SECTIONS = ['00', '01', '02']
SPLITS = ['train', 'val']

# --- Prepare Directory Structure ---
# Target: data/text_captions/ToyTrain/development
OUTPUT_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE)
OUTPUT_SUBFOLDER = 'development'
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, OUTPUT_SUBFOLDER)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Storage Location: {OUTPUT_DIR}")

# Buffers for section-specific examples
examples_per_section = {
    '00': {'source': None, 'target': None},
    '01': {'source': None, 'target': None},
    '02': {'source': None, 'target': None}
}

# --- Caption Generator Function ---
def generate_caption_from_filename(filename, section):
    """
    Parses ToyTrain metadata: HO-scale model ID, speed level, microphone
    proximity (inside/outside track), and environmental noise.
    """
    # --- A. PARAMETER EXTRACTION (REGEX) ---

    # 1. Model ID (car) - Supports alphanumeric (e.g., A1, E2)
    train_model = "A"
    car_match = re.search(r'car_([A-Z0-9]+)', filename, re.IGNORECASE)
    if car_match: train_model = car_match.group(1).upper()

    # 2. Speed Level (spd)
    speed_level = "5"
    spd_match = re.search(r'spd_(\d+)', filename, re.IGNORECASE)
    if spd_match: speed_level = spd_match.group(1)

    # 3. Microphone & Proximity Logic
    mic_id = "1"
    mic_pos = "outside" # Default positioning
    mic_match = re.search(r'mic_(\d+)', filename, re.IGNORECASE)
    if mic_match:
        mic_id = mic_match.group(1)
        # Logic: IDs 1-4 are placed outside the track, > 4 are inside
        mic_pos = "inside" if int(mic_id) > 4 else "outside"

    # 4. Environmental Noise (noise or n)
    noise_id = "1"
    noise_match = re.search(r'(?:noise|n)_(\d+)', filename, re.IGNORECASE)
    if noise_match: noise_id = noise_match.group(1)

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = f"A HO-scale ToyTrain model {train_model}"

    if is_anomaly:
        caption += " operating abnormally due to flat tire, broken axle, or track obstruction"
    else:
        caption += " operating normally"

    # --- D. TECHNICAL SPECIFICATIONS ---
    caption += f" at speed level {speed_level}"
    caption += f" recorded with microphone {mic_id} located {mic_pos} the track"
    caption += f" mixed with environmental noise pattern {noise_id}."

    return caption

# --- Main Processing Loop ---
total_files = 0

print(f"\nProcessing {MACHINE}...")

for split in SPLITS:
    # Resolve directory paths
    path_with_machine = os.path.join(SPLIT_DIR, split, MACHINE)
    path_direct = os.path.join(SPLIT_DIR, split)

    wav_dir = ""
    if os.path.exists(path_with_machine):
        wav_dir = path_with_machine
    elif os.path.exists(path_direct):
         for folder in os.listdir(path_direct):
             if folder.lower() == MACHINE.lower():
                 wav_dir = os.path.join(path_direct, folder)
                 break

    if not wav_dir:
        print(f"   Warning: Directory not found for split: {split}")
        continue

    all_wavs = glob.glob(os.path.join(wav_dir, "*.wav"))
    if not all_wavs: continue

    print(f"   Split '{split}': {len(all_wavs)} files found.")

    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '')

            # Generate semantic caption
            cap = generate_caption_from_filename(filename, section)
            captions[key] = cap

            # Track examples for logs
            if "source" in filename and examples_per_section[section]['source'] is None:
                examples_per_section[section]['source'] = f"FILE: {filename}\n      TEXT: {cap}"
            if "target" in filename and examples_per_section[section]['target'] is None:
                examples_per_section[section]['target'] = f"FILE: {filename}\n      TEXT: {cap}"

        # Persist results to JSON
        json_filename = f"captions_{MACHINE}_section_{section}_{split}.json"
        json_path = os.path.join(OUTPUT_DIR, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"     Saved: {json_filename} ({len(captions)} entries)")
        total_files += len(captions)

print("\n" + "="*40)
print(f"FINISHED: Total of {total_files} ToyTrain captions (Train/Val) created.\n")

# --- Console Summary Output ---
print("--- GENERATED EXAMPLES PER SECTION ---")
for sec in SECTIONS:
    print(f"\nSECTION {sec}:")
    ex_src = examples_per_section[sec]['source']
    print(f"   [Source]: {ex_src if ex_src else 'No files found'}")
    ex_tgt = examples_per_section[sec]['target']
    print(f"   [Target]: {ex_tgt if ex_tgt else 'No files found (normal for development split)'}")

--- Creating Captions: TOYTRAIN - TRAIN & VAL (CORRECTED MODELS) ---
Storage Location: /content/drive/MyDrive/MasterProject/data/text_captions/ToyTrain/development

Processing ToyTrain...
   Split 'train': 2727 files found.
     Saved: captions_ToyTrain_section_00_train.json (909 entries)
     Saved: captions_ToyTrain_section_01_train.json (909 entries)
     Saved: captions_ToyTrain_section_02_train.json (909 entries)
   Split 'val': 273 files found.
     Saved: captions_ToyTrain_section_00_val.json (91 entries)
     Saved: captions_ToyTrain_section_01_val.json (91 entries)
     Saved: captions_ToyTrain_section_02_val.json (91 entries)

FINISHED: Total of 3000 ToyTrain captions (Train/Val) created.

--- GENERATED EXAMPLES PER SECTION ---

SECTION 00:
   [Source]: FILE: section_00_source_train_normal_0972_car_E2_spd_8_mic_1_noise_1.wav
      TEXT: A HO-scale ToyTrain model E2 operating normally at speed level 8 recorded with microphone 1 located outside the track mixed with environmenta

In [ ]:
# ============================================================
# SCRIPT: Generate Captions for TOYTRAIN - ADDITIONAL (03-05)
# Logic: Processing Additional Training Data partitions for ToyTrain
# ============================================================

print("--- Creating Captions: TOYTRAIN - ADDITIONAL (03-05) ---")

# --- Configuration ---
BASE_DIR = '/content/drive/MyDrive/MasterProject'
SPLIT_DIR = os.path.join(BASE_DIR, 'data/splits')

MACHINE = 'ToyTrain'
SECTIONS = ['03', '04', '05']
SPLITS = ['train_additional', 'val_additional']

# --- Prepare Directory Structure ---
# Target: data/text_captions/ToyTrain/additional
OUTPUT_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE)
OUTPUT_SUBFOLDER = 'additional'
OUTPUT_DIR = os.path.join(OUTPUT_ROOT, OUTPUT_SUBFOLDER)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Storage Location: {OUTPUT_DIR}")

# Buffers for section-specific examples
examples_per_section = {
    '03': {'source': None, 'target': None},
    '04': {'source': None, 'target': None},
    '05': {'source': None, 'target': None}
}

# --- Caption Generator Function ---
def generate_caption_from_filename(filename, section):
    """
    Parses ToyTrain metadata: HO-scale model ID, speed level,
    microphone placement (inside/outside), and environmental noise.
    """
    # --- A. PARAMETER EXTRACTION (REGEX) ---

    # 1. Model ID (car) - Supports alphanumeric (e.g., A1)
    train_model = "A"
    car_match = re.search(r'car_([A-Z0-9]+)', filename, re.IGNORECASE)
    if car_match: train_model = car_match.group(1).upper()

    # 2. Speed Level (spd) - Integer values 5-9
    speed_level = "5"
    spd_match = re.search(r'spd_(\d+)', filename, re.IGNORECASE)
    if spd_match: speed_level = spd_match.group(1)

    # 3. Microphone & Position Logic
    mic_id = "1"
    mic_pos = "outside" # Default (mics 1-4)

    mic_match = re.search(r'mic_(\d+)', filename, re.IGNORECASE)
    if mic_match:
        mic_id = mic_match.group(1)
        # Logic: mics 1-4 are outside the loop, mics 5-8 are inside
        mic_pos = "inside" if int(mic_id) > 4 else "outside"

    # 4. Environmental Noise (noise)
    noise_id = "1"
    noise_match = re.search(r'(?:noise|n)_(\d+)', filename, re.IGNORECASE)
    if noise_match: noise_id = noise_match.group(1)

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = f"A HO-scale ToyTrain model {train_model}"

    if is_anomaly:
        # Detailed anomaly descriptions based on dataset documentation
        caption += " operating abnormally due to flat tire, broken axle, or track obstruction"
    else:
        caption += " operating normally"

    # --- D. TECHNICAL SPECIFICATIONS ---
    caption += f" at speed level {speed_level}"
    caption += f" recorded with microphone {mic_id} located {mic_pos} the track"
    caption += f" mixed with environmental noise pattern {noise_id}."

    return caption

# --- Main Processing Loop ---
total_files = 0

print(f"\nProcessing {MACHINE}...")

for split_folder in SPLITS:
    # Resolve directory path
    path_with_machine = os.path.join(SPLIT_DIR, split_folder, MACHINE)
    path_direct = os.path.join(SPLIT_DIR, split_folder)

    wav_dir = ""
    if os.path.exists(path_with_machine):
        wav_dir = path_with_machine
    elif os.path.exists(path_direct):
         for folder in os.listdir(path_direct):
             if folder.lower() == MACHINE.lower():
                 wav_dir = os.path.join(path_direct, folder)
                 break

    if not wav_dir:
        print(f"   Warning: Directory not found for split: {split_folder}")
        continue

    all_wavs = glob.glob(os.path.join(wav_dir, "*.wav"))
    if not all_wavs: continue

    print(f"   Directory '{split_folder}': {len(all_wavs)} files found.")

    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '')

            # Generate semantic caption
            cap = generate_caption_from_filename(filename, section)
            captions[key] = cap

            # Track examples for logs
            if "source" in filename and examples_per_section[section]['source'] is None:
                examples_per_section[section]['source'] = f"FILE: {filename}\n      TEXT: {cap}"
            if "target" in filename and examples_per_section[section]['target'] is None:
                examples_per_section[section]['target'] = f"FILE: {filename}\n      TEXT: {cap}"

        # Persist results to JSON
        json_filename = f"captions_{MACHINE}_section_{section}_{split_folder}.json"
        json_path = os.path.join(OUTPUT_DIR, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"     Saved: {json_filename} ({len(captions)} entries)")
        total_files += len(captions)

print("\n" + "="*40)
print(f"FINISHED: Total of {total_files} ToyTrain captions (Additional) created.\n")

# --- Console Summary Output ---
print("--- GENERATED EXAMPLES PER SECTION ---")
for sec in SECTIONS:
    print(f"\nSECTION {sec}:")
    ex_src = examples_per_section[sec]['source']
    print(f"   [Source]: {ex_src if ex_src else 'No files found'}")
    ex_tgt = examples_per_section[sec]['target']
    print(f"   [Target]: {ex_tgt if ex_tgt else 'No files found'}")

--- Creating Captions: TOYTRAIN - ADDITIONAL (03-05) ---
Storage Location: /content/drive/MyDrive/MasterProject/data/text_captions/ToyTrain/additional

Processing ToyTrain...
   Directory 'train_additional': 2730 files found.
     Saved: captions_ToyTrain_section_03_train_additional.json (910 entries)
     Saved: captions_ToyTrain_section_04_train_additional.json (910 entries)
     Saved: captions_ToyTrain_section_05_train_additional.json (910 entries)
   Directory 'val_additional': 270 files found.
     Saved: captions_ToyTrain_section_03_val_additional.json (90 entries)
     Saved: captions_ToyTrain_section_04_val_additional.json (90 entries)
     Saved: captions_ToyTrain_section_05_val_additional.json (90 entries)

FINISHED: Total of 3000 ToyTrain captions (Additional) created.

--- GENERATED EXAMPLES PER SECTION ---

SECTION 03:
   [Source]: FILE: section_03_source_train_normal_0306_car_H1_spd_7_mic_1_noise_6.wav
      TEXT: A HO-scale ToyTrain model H1 operating normally at speed 

In [ ]:
import os
import re
import glob
import json

# ============================================================
# SCRIPT: Generate Captions for TOYTRAIN - TEST DATA (SPLIT)
# Logic: Processing 'test_train' and 'test_val' folders separately
#        Extracts Model ID, Speed, Mic Position, and Noise Pattern
# ============================================================

print("--- Creating Captions: TOYTRAIN - TEST SPLITS (TRAIN/VAL) ---")

# 1. Configuration
BASE_DIR = '/content/drive/MyDrive/MasterProject'

# Root of the split audio files
TEST_SPLIT_ROOT = os.path.join(BASE_DIR, 'data/splits/test')

# The two sub-splits to process
SUB_SPLITS = ['test_train', 'test_val']
MACHINE = 'ToyTrain'
SECTIONS = ['00', '01', '02']

# Target Base Directory for Captions
# Structure will be: data/text_captions/ToyTrain/test/test_train/ etc.
CAPTION_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions', MACHINE, 'test')

# Global variables for console examples
example_source = None
example_target = None

# 2. Caption Generator Function (ToyTrain-Specific)
def generate_test_caption(filename, section):
    """
    Parses ToyTrain test metadata: HO-scale model ID, speed level,
    microphone proximity logic, and environmental noise patterns.
    """
    # --- A. PARAMETER EXTRACTION ---

    # 1. Model ID (e.g., A, A1, B2)
    train_model = "A"
    car_match = re.search(r'car_([A-Z0-9]+)', filename, re.IGNORECASE)
    if car_match: train_model = car_match.group(1).upper()

    # 2. Speed Level (spd) - Integer values (5-9)
    speed_level = "5"
    spd_match = re.search(r'spd_(\d+)', filename, re.IGNORECASE)
    if spd_match: speed_level = spd_match.group(1)

    # 3. Microphone & Position Logic
    mic_id = "1"
    mic_pos = "outside" # Default (mics 1-4)

    mic_match = re.search(r'mic_(\d+)', filename, re.IGNORECASE)
    if mic_match:
        mic_id = mic_match.group(1)
        # Logic: mics 1-4 = outside track, mics 5-8 = inside track
        if int(mic_id) > 4:
            mic_pos = "inside"
        else:
            mic_pos = "outside"

    # 4. Environmental Noise (noise)
    noise_id = "1"
    noise_match = re.search(r'(?:noise|n)_(\d+)', filename, re.IGNORECASE)
    if noise_match: noise_id = noise_match.group(1)

    # --- B. STATUS IDENTIFICATION ---
    is_anomaly = "anomaly" in filename

    # --- C. TEXT CONSTRUCTION ---
    caption = f"A HO-scale ToyTrain model {train_model}"

    if is_anomaly:
        caption += " operating abnormally"
    else:
        caption += " operating normally"

    # --- D. TECHNICAL SPECIFICATIONS ---
    caption += f" at speed level {speed_level}"
    caption += f" recorded with microphone {mic_id} located {mic_pos} the track"
    caption += f" mixed with environmental noise pattern {noise_id}."

    return caption

# 3. MAIN PROCESSING LOOP (Iterates through splits)
for sub_split in SUB_SPLITS:

    # Input Path: .../splits/test/test_train/ToyTrain
    # Handle mixed case folder naming (ToyTrain vs toytrain)
    input_base = os.path.join(TEST_SPLIT_ROOT, sub_split)
    input_dir = None

    if os.path.exists(input_base):
        for folder in os.listdir(input_base):
            if folder.lower() == MACHINE.lower():
                input_dir = os.path.join(input_base, folder)
                break

    # Output Path: .../text_captions/ToyTrain/test/test_train
    output_dir = os.path.join(CAPTION_ROOT, sub_split)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n📂 Processing Split: {sub_split.upper()}")
    print(f"   Input:  {input_dir}")
    print(f"   Output: {output_dir}")

    if not input_dir or not os.path.exists(input_dir):
        print(f"   ⚠️ Warning: Directory not found: {input_dir}")
        continue

    # Find all .wav files
    all_wavs = glob.glob(os.path.join(input_dir, "*.wav"))

    if not all_wavs:
        print("   ⚠️ No WAV files found.")
        continue

    print(f"   Found {len(all_wavs)} audio files.")

    total_files_split = 0

    # Process by Section
    for section in SECTIONS:
        section_str = f"section_{section}"
        sec_files = [f for f in all_wavs if section_str in os.path.basename(f)]

        if not sec_files: continue

        captions = {}
        for filepath in sec_files:
            filename = os.path.basename(filepath)
            key = filename.replace('.wav', '') # JSON Key

            # Generate semantic text
            cap = generate_test_caption(filename, section)
            captions[key] = cap

            # Track examples for verification (Only from train split)
            if sub_split == 'test_train':
                if example_source is None and "source" in filename:
                    example_source = f"[{sub_split}] SOURCE: {filename}\n   TEXT: {cap}"
                if example_target is None and "target" in filename:
                    example_target = f"[{sub_split}] TARGET: {filename}\n   TEXT: {cap}"

        # Save JSON (Filename includes split name)
        json_filename = f"captions_{MACHINE}_section_{section}_{sub_split}.json"
        json_path = os.path.join(output_dir, json_filename)

        with open(json_path, 'w') as f:
            json.dump(captions, f, indent=4)

        print(f"   ✅ Saved: {json_filename} ({len(captions)} entries)")
        total_files_split += len(captions)

print("\n" + "="*40)
print("FINISHED TOYTRAIN CAPTION GENERATION.")

# --- CONSOLE OUTPUT EXAMPLES ---
print("--- GENERATED EXAMPLES ---")
if example_source:
    print(example_source)
    print("-" * 20)
if example_target:
    print(example_target)
else:
    print("WARNING: No Target domain example found.")

--- Creating Captions: TOYTRAIN - TEST SPLITS (TRAIN/VAL) ---

📂 Processing Split: TEST_TRAIN
   Input:  /content/drive/MyDrive/MasterProject/data/splits/test/test_train/ToyTrain
   Output: /content/drive/MyDrive/MasterProject/data/text_captions/ToyTrain/test/test_train
   Found 540 audio files.
   ✅ Saved: captions_ToyTrain_section_00_test_train.json (180 entries)
   ✅ Saved: captions_ToyTrain_section_01_test_train.json (180 entries)
   ✅ Saved: captions_ToyTrain_section_02_test_train.json (180 entries)

📂 Processing Split: TEST_VAL
   Input:  /content/drive/MyDrive/MasterProject/data/splits/test/test_val/ToyTrain
   Output: /content/drive/MyDrive/MasterProject/data/text_captions/ToyTrain/test/test_val
   Found 60 audio files.
   ✅ Saved: captions_ToyTrain_section_00_test_val.json (20 entries)
   ✅ Saved: captions_ToyTrain_section_01_test_val.json (20 entries)
   ✅ Saved: captions_ToyTrain_section_02_test_val.json (20 entries)

FINISHED TOYTRAIN CAPTION GENERATION.
--- GENERATED EXAMP

In [ ]:
import os
import glob
import json
import torch
from tqdm.auto import tqdm
from transformers import T5Tokenizer, T5EncoderModel

print("--- GENERATING EMBEDDINGS (DIRECTLY INTO CORRECT FOLDERS) ---")

# ==========================================
# 1. KONFIGURATION
# ==========================================
BASE_DIR = '/content/drive/MyDrive/MasterProject'
SPLITS_DIR = os.path.join(BASE_DIR, 'data', 'splits')
CAPTION_ROOT = os.path.join(BASE_DIR, 'data', 'text_captions')
EMBEDDING_ROOT = os.path.join(BASE_DIR, 'data', 'text_embeddings')

MACHINES = ['bearing', 'fan', 'gearbox', 'slider', 'valve', 'ToyCar', 'ToyTrain']

# HIER DEFINIEREN WIR DIE HARTE ZUORDNUNG
# Links: Wo liegen die WAVs (in data/splits)?
# Rechts: Wo sollen die Embeddings hin (in data/text_embeddings/Maschine/...)?
SPLIT_MAPPING = [
    # (Split-Ordner Name,   Ziel-Ordner Struktur)
    ('train',              'development/train'),
    ('val',                'development/val'),
    ('test/test_train',    'test/test_train'),
    ('test/test_val',      'test/test_val'),
    # Optional: Falls du Additional hast
    ('train_additional',   'additional/train'),
    ('val_additional',     'additional/val')
]

MAX_SEQ_LENGTH = 64
MODEL_NAME = "google/flan-t5-base"

# ==========================================
# 2. MODEL LADEN
# ==========================================
print(f"⏳ Lade Model: {MODEL_NAME}...")
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5EncoderModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
print(f"✅ Model bereit.")

# ==========================================
# 3. HELPER: Captions laden
# ==========================================
def load_all_captions_for_machine(machine):
    """Lädt ALLE JSONs einer Maschine in ein einziges großes Dictionary."""
    all_captions = {}
    # Suche rekursiv nach allen JSONs für diese Maschine
    json_pattern = os.path.join(CAPTION_ROOT, machine, '**', '*.json')
    json_files = glob.glob(json_pattern, recursive=True)

    for jf in json_files:
        with open(jf, 'r') as f:
            data = json.load(f)
            # data ist z.B. {'file_001': 'caption...', 'file_002': ...}
            all_captions.update(data)
    return all_captions

def create_embedding(text):
    with torch.no_grad():
        inputs = tokenizer(text, return_tensors="pt", padding="max_length", truncation=True, max_length=MAX_SEQ_LENGTH).to(device)
        return model(**inputs).last_hidden_state.cpu()

# ==========================================
# 4. MAIN LOOP
# ==========================================
total_files = 0

for machine in MACHINES:
    print(f"\n📂 Bearbeite Maschine: {machine}")

    # 1. Erst alle Texte laden (damit wir sie parat haben)
    print("   Lade Captions in den Speicher...")
    caption_db = load_all_captions_for_machine(machine)
    if not caption_db:
        print("   ⚠️ Keine Captions gefunden! Überspringe Maschine.")
        continue

    # 2. Durch die definierten Splits gehen
    for source_split, target_subpath in SPLIT_MAPPING:

        # Wo sind die WAVs? (z.B. data/splits/train/fan)
        wav_dir = os.path.join(SPLITS_DIR, source_split, machine)

        # Wo soll das Embedding hin? (z.B. data/text_embeddings/fan/development/train)
        target_dir = os.path.join(EMBEDDING_ROOT, machine, target_subpath)

        if not os.path.exists(wav_dir):
            continue

        # Zielordner erstellen
        os.makedirs(target_dir, exist_ok=True)

        # Alle WAVs in diesem Split finden
        wav_files = glob.glob(os.path.join(wav_dir, "*.wav"))
        if not wav_files:
            continue

        print(f"   -> Generiere Embeddings für '{target_subpath}' ({len(wav_files)} Dateien)")

        for wav_path in tqdm(wav_files, desc=f"   Encoding {target_subpath}", leave=False):
            filename = os.path.basename(wav_path)
            key = filename.replace('.wav', '') # Key für Caption Lookup

            # Text holen
            if key in caption_db:
                caption_text = caption_db[key]

                # Embedding machen
                emb = create_embedding(caption_text)

                # Speichern (direkt im richtigen Ordner!)
                save_path = os.path.join(target_dir, key + ".pt")
                torch.save(emb, save_path)
                total_files += 1
            else:
                # Falls keine Caption da ist (sollte nicht passieren)
                # print(f"Warnung: Keine Caption für {filename}")
                pass

print("\n" + "="*40)
print(f"✅ FERTIG: {total_files} Embeddings sauber generiert.")

# CHECK
test_path = os.path.join(EMBEDDING_ROOT, 'fan', 'development', 'train')
if os.path.exists(test_path):
    print(f"\nCheck {test_path}:")
    print(f"Enthält {len(os.listdir(test_path))} Dateien.")

--- GENERATING EMBEDDINGS (DIRECTLY INTO CORRECT FOLDERS) ---
⏳ Lade Model: google/flan-t5-base...
✅ Model bereit.

📂 Bearbeite Maschine: bearing
   Lade Captions in den Speicher...
   -> Generiere Embeddings für 'development/train' (2726 Dateien)


   Encoding development/train:   0%|          | 0/2726 [00:00<?, ?it/s]

   -> Generiere Embeddings für 'development/val' (273 Dateien)


   Encoding development/val:   0%|          | 0/273 [00:00<?, ?it/s]

   -> Generiere Embeddings für 'test/test_train' (540 Dateien)


...

   -> Generiere Embeddings für 'additional/train' (2730 Dateien)


   Encoding additional/train:   0%|          | 0/2730 [00:00<?, ?it/s]

   -> Generiere Embeddings für 'additional/val' (270 Dateien)


   Encoding additional/val:   0%|          | 0/270 [00:00<?, ?it/s]


✅ FERTIG: 46199 Embeddings sauber generiert.

Check /content/drive/MyDrive/MasterProject/data/text_embeddings/fan/development/train:
Enthält 2727 Dateien.


In [ ]:
import os
import glob
import torch
import numpy as np
from tqdm.auto import tqdm
from google.colab import drive

# 1. Drive mounten
drive.mount('/content/drive')

# --- ANGEPASSTE PFADE ---
# Pfad zu den Roh-Latents
SOURCE_PATH = '/content/drive/MyDrive/MasterProject/data/features/encodec'
# Speicherort für die Statistik (ebenfalls im features-Ordner)
SAVE_PATH = '/content/drive/MyDrive/MasterProject/data/features/bearing_global_stats.pt'

# Deine Unterordner-Struktur
SUB_DIRS = [
    'train',
    'train_additional',
    'val',
    'val_additional',
    'test/test_train',
    'test/test_val'
]

def calculate_and_save_stats():
    all_files = []
    print(f"📂 Durchsuche: {SOURCE_PATH}")

    for sd in SUB_DIRS:
        folder = os.path.join(SOURCE_PATH, sd)
        if os.path.exists(folder):
            # Finde alle .pt Dateien und filtere nach 'bearing'
            files = glob.glob(os.path.join(folder, "**", "*.pt"), recursive=True)
            bearing_files = [f for f in files if 'bearing' in f.lower()]
            all_files.extend(bearing_files)
            print(f"   ✅ {sd}: {len(bearing_files)} Dateien gefunden")

    if not all_files:
        print("❌ Keine Dateien gefunden! Bitte prüfe, ob SOURCE_PATH korrekt ist.")
        return

    # Statistik-Variablen für 16 Kanäle
    sum_l = torch.zeros(16)
    sum_sq_l = torch.zeros(16)
    total_elements = 0

    print(f"\n🔢 Starte Berechnung für {len(all_files)} Dateien auf CPU...")

    for f in tqdm(all_files, desc="Berechne"):
        try:
            # Laden auf CPU (schont GPU-Ressourcen)
            latent = torch.load(f, map_location='cpu')

            # Dimensionen anpassen (EnCodec [1, 128, T] -> [16, 8, T])
            if latent.dim() == 3:
                latent = latent.squeeze(0)
            latent = latent.view(16, 8, -1)

            # Anzahl der Datenpunkte (Kanäle x Bins x Zeit)
            n_per_channel = latent.shape[1] * latent.shape[2]

            # Summen pro Kanal akkumulieren
            sum_l += latent.sum(dim=(1, 2))
            sum_sq_l += (latent**2).sum(dim=(1, 2))
            total_elements += n_per_channel
        except Exception as e:
            continue

    # Mittelwert (Mean) und Standardabweichung (Std) pro Kanal berechnen
    global_mean = sum_l / total_elements
    global_std = torch.sqrt((sum_sq_l / total_elements) - global_mean**2 + 1e-6)

    # Ergebnisse als Dictionary speichern
    stats = {
        'mean': global_mean,
        'std': global_std,
        'info': 'Globale Statistik für Bearing (train, val, test, additional)'
    }

    torch.save(stats, SAVE_PATH)

    print("\n" + "="*50)
    print("🏆 STATISTIK ERFOLGREICH GESPEICHERT!")
    print(f"Datei: {SAVE_PATH}")
    print("-" * 50)
    print(f"Mittelwerte (Erste 3 Kanäle): {global_mean[:3].numpy()}")
    print(f"Standardabweichungen (Erste 3 Kanäle): {global_std[:3].numpy()}")
    print("="*50)

# Ausführen
calculate_and_save_stats()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Durchsuche: /content/drive/MyDrive/MasterProject/data/features/encodec
   ✅ train: 2726 Dateien gefunden
   ✅ train_additional: 2730 Dateien gefunden
   ✅ val: 273 Dateien gefunden
   ✅ val_additional: 270 Dateien gefunden
   ✅ test/test_train: 540 Dateien gefunden
   ✅ test/test_val: 60 Dateien gefunden

🔢 Starte Berechnung für 6599 Dateien auf CPU...


Berechne:   0%|          | 0/6599 [00:00<?, ?it/s]


🏆 STATISTIK ERFOLGREICH GESPEICHERT!
Datei: /content/drive/MyDrive/MasterProject/data/features/bearing_global_stats.pt
--------------------------------------------------
Mittelwerte (Erste 3 Kanäle): [ 0.9170957  -0.88777775 -2.0908551 ]
Standardabweichungen (Erste 3 Kanäle): [4.339158  3.3051305 3.8344982]
